# Proyecto final — Red de transferencias de Ethereum

**Estructura de Datos — Universidad Industrial de Santander**

**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros

---

Este notebook usa **un solo dataset**:

```
transferencias.parquet
```

El profesor lo sube y el notebook hace todo lo demás:

1. Analiza las 38,9 M de transferencias con DuckDB.
2. Genera una muestra conectada de USDC.
3. Ejecuta las cuatro tareas con estructuras de datos básicas.
4. Explica la teoría y la complejidad de cada algoritmo.

| Tarea | Pregunta | Algoritmo | Responsable |
|---|---|---|---|
| 1 | ¿Cuál es el camino más barato? | Bellman-Ford | Pablo Álvarez |
| 2 | ¿Qué nodos son más importantes? | Grado + PageRank | Jhon Arciniegas |
| 3 | ¿Qué grupos de nodos hay? | Louvain | Emerson Ballesteros |
| 4 | ¿Cuál es la red mínima? | Kruskal + Union-Find | Grupal |

> **Cómo ejecutar:** sube `transferencias.parquet`, ejecuta las celdas en
> orden y listo. No necesitas clonar el repositorio ni instalar módulos.

## Teoría: grafos y estructuras de datos

Un **grafo** tiene nodos y aristas. En este proyecto:

- **Nodo**: una dirección de Ethereum.
- **Arista**: una transferencia de una dirección a otra.
- **Peso**: la cantidad transferida.

El grafo es **dirigido** (A → B no es lo mismo que B → A) y **ponderado**.

### Estructuras de datos que usamos

| Estructura | Dónde se usa | Para qué |
|---|---|---|
| **Lista de adyacencia** (`dict` de listas) | T1, T2, T3 | recorrer los vecinos de un nodo |
| **Diccionario** | T1, T2, T3 | guardar distancias, puntajes y comunidades |
| **Conjunto** (`set`) | T2, T3 | nodos visitados y comunidades vecinas |
| **Cola** (`deque`) | T1, T3 | recorrer por niveles y encontrar componentes |
| **Lista de aristas** | T4 | ordenar las aristas por peso |
| **Union-Find** | T4 | detectar ciclos en Kruskal |

La idea es simple: **cada algoritmo usa la estructura que le permite hacer su trabajo sin recorrer el grafo de más**.

## 0. Capa de datos: el grafo

Esta celda define la estructura compartida: un grafo dirigido y ponderado.

- **Nodos**: direcciones Ethereum.
- **Aristas**: transferencias `(from_address, to_address, value)`.
- **Peso**: cantidad transferida.

El resto de celdas usan el mismo objeto `grafo`.

In [ ]:
"""
grafo.py
========
Carga del dataset y construcción del grafo.

En este proyecto modelamos las transferencias de un token ERC-20 de Ethereum
como un grafo DIRIGIDO y PONDERADO G = (V, E):

    - V (vértices): las direcciones (wallets, exchanges, contratos).
    - E (aristas): las transferencias de una dirección a otra.
    - w(e) (peso): la cantidad transferida.

La carga acepta **CSV** y **Parquet**. Para Parquet se usa DuckDB solo para
leer y muestrear el archivo; los algoritmos del proyecto siguen siendo Python
puro y reciben el grafo ya construido.

Solo usamos estructuras nativas de Python: diccionarios y listas.
"""

from collections import defaultdict, deque
from typing import Dict, List, Optional, Set, Tuple

# Lista de adyacencia: cada nodo apunta a una lista de (vecino, peso).
AdjList = Dict[str, List[Tuple[str, float]]]

# Nombres de columna que aceptamos para cada dato. Se comparan sin distinguir
# mayúsculas/minúsculas, así funciona con distintos exports del dataset.
COLUMNAS = {
    "from": ["from_address", "from", "sender", "from_addr", "fromaddress"],
    "to": ["to_address", "to", "receiver", "to_addr", "toaddress"],
    "value": ["value", "amount", "token_amount", "quantity", "val"],
    "token": ["token_address", "contract_address", "token", "tokenaddress"],
    "time": ["block_timestamp", "timestamp", "block_time", "date", "time"],
    "block": ["block_number", "block"],
}

DIRECCION_CERO = "0x0000000000000000000000000000000000000000"
DIRECCIONES_CERO = {DIRECCION_CERO, "0x0", "0x00"}


def _elegir_columnas(disponibles: list[str]) -> Dict[str, Optional[str]]:
    """Mapea cada dato lógico (from, to, value...) a su columna real.

    `disponibles` debe venir en minúsculas y en el orden del archivo; el
    resultado conserva el nombre real de la columna para poder citarlo en SQL.
    """
    return {
        clave: next((opcion for opcion in opciones if opcion in disponibles), None)
        for clave, opciones in COLUMNAS.items()
    }


def _sql_texto(valor: str) -> str:
    """Escapa un texto para usarlo dentro de una consulta SQL."""
    return "'" + str(valor).replace("'", "''") + "'"


def _sql_columna(columna: str) -> str:
    """Cita un identificador de columna para SQL."""
    return '"' + str(columna).replace('"', '""') + '"'


class Grafo:
    """Grafo dirigido y ponderado representado con una lista de adyacencia."""

    def __init__(self) -> None:
        # {origen: [(destino, peso), ...]}
        self.adj: AdjList = defaultdict(list)
        # Todos los vértices del grafo.
        self.nodos: Set[str] = set()
        # Lista de aristas como (origen, destino, peso).
        self.aristas: List[Tuple[str, str, float]] = []
        # {dirección: {"nombre": ..., "tipo": ...}} para mostrar resultados legibles.
        self.etiquetas: Dict[str, Dict[str, str]] = {}

    # Carga de datos

    def _agregar_arista(
        self,
        origen: str,
        destino: str,
        peso: float,
        limpiar: bool = True,
    ) -> bool:
        """Agrega una arista y devuelve si fue aceptada.

        Con `limpiar=True` se descartan transferencias inválidas para el
        análisis: dirección cero, auto-transferencias y valores no positivos.
        """
        origen = str(origen).strip().lower()
        destino = str(destino).strip().lower()
        peso = float(peso)

        if limpiar:
            if not origen or not destino:
                return False
            if origen == destino:
                return False
            if origen in DIRECCIONES_CERO or destino in DIRECCIONES_CERO:
                return False
            if peso <= 0:
                return False

        self.nodos.add(origen)
        self.nodos.add(destino)
        self.adj[origen].append((destino, peso))
        self.aristas.append((origen, destino, peso))
        return True

    def cargar_transferencias(
        self,
        ruta_csv: str,
        max_filas: Optional[int] = None,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un CSV de transferencias y llena la lista de adyacencia.

        El CSV debe tener (al menos) columnas equivalentes a:
            from_address, to_address, value
        (se aceptan alias como `from`, `sender`, `to`, `receiver`, etc.).

        Parámetros:
            ruta_csv: ruta al archivo CSV.
            max_filas: opcional. Leer solo las primeras `max_filas` filas
                       (útil para probar con una muestra).
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        import csv

        with open(ruta_csv, newline="", encoding="utf-8") as archivo:
            lector = csv.DictReader(archivo)
            campos = lector.fieldnames or []
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            col_from = reales[mapa["from"]]
            col_to = reales[mapa["to"]]
            col_value = reales[mapa["value"]]

            for i, fila in enumerate(lector):
                if max_filas is not None and i >= max_filas:
                    break
                self._agregar_arista(
                    fila[col_from],
                    fila[col_to],
                    fila[col_value],
                    limpiar=limpiar,
                )

        return self

    def cargar_parquet(
        self,
        ruta: str,
        max_filas: Optional[int] = None,
        muestra: Optional[int] = None,
        token: Optional[str] = None,
        desde: Optional[str] = None,
        hasta: Optional[str] = None,
        semilla: int = 42,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un Parquet (o una muestra) usando DuckDB.

        DuckDB solo se encarga de leer, filtrar y muestrear; el grafo se
        construye igual que con el CSV, usando Python puro.

        Parámetros:
            ruta: ruta al archivo Parquet.
            max_filas: leer solo las primeras N filas.
            muestra: tomar una muestra aleatoria de N filas (recomendado para
                     el dataset completo).
            token: filtrar por `token_address`.
            desde / hasta: filtrar por `block_timestamp` (texto ISO) si existe.
            semilla: semilla de la muestra aleatoria.
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        try:
            import duckdb
        except ImportError as exc:  # pragma: no cover - depende del entorno
            raise SystemExit(
                "Falta duckdb para leer Parquet. Instálalo con:\n"
                "    pip install -r requirements-datos.txt"
            ) from exc

        con = duckdb.connect()
        try:
            campos = list(con.read_parquet(ruta).columns)
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            select = (
                f"CAST({_sql_columna(reales[mapa['from']])} AS VARCHAR) AS from_address, "
                f"CAST({_sql_columna(reales[mapa['to']])} AS VARCHAR) AS to_address, "
                f"CAST({_sql_columna(reales[mapa['value']])} AS DOUBLE) AS value"
            )

            filtros: list[str] = []
            if token:
                if not mapa["token"]:
                    raise ValueError("Pediste `token` pero el Parquet no tiene columna de token.")
                filtros.append(
                    f"CAST({_sql_columna(reales[mapa['token']])} AS VARCHAR) = "
                    f"{_sql_texto(token)}"
                )
            if desde or hasta:
                if not mapa["time"]:
                    raise ValueError(
                        "Pediste una ventana de tiempo pero el Parquet no tiene "
                        "columna de tiempo (block_timestamp)."
                    )
                col_tiempo = _sql_columna(reales[mapa["time"]])
                if desde:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) >= {_sql_texto(desde)}")
                if hasta:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) <= {_sql_texto(hasta)}")

            consulta = f"SELECT {select} FROM read_parquet({_sql_texto(ruta)})"
            if filtros:
                consulta += " WHERE " + " AND ".join(filtros)

            if muestra is not None:
                n = int(muestra)
                consulta += f" USING SAMPLE reservoir({n} ROWS)"
                if semilla is not None:
                    consulta += f" REPEATABLE ({int(semilla)})"
            elif max_filas is not None:
                consulta += f" LIMIT {int(max_filas)}"

            try:
                filas = con.execute(consulta).fetchall()
            except Exception as exc:  # DuckDB antiguo sin REPEATABLE/reservoir
                mensaje = str(exc).lower()
                if muestra is not None and ("parser" in mensaje or "syntax" in mensaje):
                    base = consulta.split(" USING SAMPLE ")[0]
                    filas = con.execute(f"{base} LIMIT {int(muestra)}").fetchall()
                else:
                    raise
        finally:
            con.close()

        for origen, destino, peso in filas:
            self._agregar_arista(origen, destino, peso, limpiar=limpiar)

        return self

    def cargar(self, ruta: str, max_filas: Optional[int] = None, **opciones) -> "Grafo":
        """Carga transferencias detectando el formato por la extensión.

        - `.csv`  -> `cargar_transferencias`
        - `.parquet` / `.pq` -> `cargar_parquet`

        Las opciones específicas de Parquet (`muestra`, `token`, `desde`,
        `hasta`, `semilla`, `limpiar`) se pasan tal cual.
        """
        if str(ruta).lower().endswith((".parquet", ".pq")):
            return self.cargar_parquet(ruta, max_filas=max_filas, **opciones)

        sobrantes = set(opciones) - {"limpiar"}
        if sobrantes:
            raise ValueError(
                f"Las opciones {sorted(sobrantes)} solo aplican a archivos Parquet."
            )
        return self.cargar_transferencias(ruta, max_filas=max_filas, **opciones)

    def cargar_etiquetas(self, ruta_csv: str) -> "Grafo":
        """Lee un CSV con el nombre legible de cada dirección.

        El CSV debe tener las columnas: address, name, type.
        """
        import csv

        # Algunos exports de etiquetas vienen en latin-1 (nombres con tildes).
        filas = None
        for codificacion in ("utf-8", "latin-1"):
            try:
                with open(ruta_csv, newline="", encoding=codificacion) as archivo:
                    filas = list(csv.DictReader(archivo))
                break
            except UnicodeDecodeError:
                continue
        if filas is None:  # pragma: no cover - archivo binario inválido
            raise ValueError(f"No se pudo leer el CSV de etiquetas: {ruta_csv}")

        for fila in filas:
            addr = fila["address"].strip().lower()
            self.etiquetas[addr] = {
                "nombre": fila.get("name", "").strip(),
                "tipo": fila.get("type", "").strip(),
            }
        return self

    # Utilidades

    def conservar_componente_mayor(self) -> "Grafo":
        """Deja solo la componente conexa (no dirigida) más grande.

        Un muestreo aleatorio deja muchos nodos aislados. Quedarse con la
        componente más grande produce una red conectada y resultados
        interpretables, sin volver a leer el archivo.

        Retorna:
            self, para poder encadenar llamadas.
        """
        adyacencia: dict[str, set[str]] = defaultdict(set)
        for origen, destino, _ in self.aristas:
            adyacencia[origen].add(destino)
            adyacencia[destino].add(origen)

        visitados: Set[str] = set()
        mejor: Set[str] = set()
        for inicio in adyacencia:
            if inicio in visitados:
                continue
            componente: Set[str] = set()
            cola = deque([inicio])
            visitados.add(inicio)
            while cola:
                nodo = cola.popleft()
                componente.add(nodo)
                for vecino in adyacencia[nodo]:
                    if vecino not in visitados:
                        visitados.add(vecino)
                        cola.append(vecino)
            if len(componente) > len(mejor):
                mejor = componente

        if not mejor or len(mejor) == len(self.nodos):
            return self

        aristas = [
            (a, b, w) for a, b, w in self.aristas if a in mejor and b in mejor
        ]
        self.adj = defaultdict(list)
        self.nodos = set()
        self.aristas = []
        for origen, destino, peso in aristas:
            self._agregar_arista(origen, destino, peso, limpiar=False)
        return self

    def nombre(self, addr: str) -> str:
        """Nombre legible de un nodo, o la propia dirección si no está etiquetada."""
        return self.etiquetas.get(addr, {}).get("nombre", addr)

    def resumen(self) -> Dict[str, float]:
        """Estadísticas básicas: número de nodos, aristas, densidad y peso."""
        n = len(self.nodos)
        m = len(self.aristas)
        # Densidad de un grafo dirigido: |E| / (|V| * (|V| - 1))
        densidad = m / (n * (n - 1)) if n > 1 else 0.0
        peso_total = sum(w for _, _, w in self.aristas)
        return {
            "nodos": n,
            "aristas": m,
            "densidad": round(densidad, 4),
            "peso_total": round(peso_total, 2),
        }


def cargar_grafo(
    ruta: str,
    max_filas: Optional[int] = None,
    etiquetas: Optional[str] = None,
    limpiar: bool = True,
    **opciones,
) -> Grafo:
    """Carga el dataset UNA vez y devuelve el objeto compartido `Grafo`.

    Es la función que usa el notebook. Las cuatro tareas reciben
    `grafo.aristas` y construyen su propia estructura; no vuelven a leer el
    archivo.

    Por defecto limpia transferencias inválidas (dirección cero, autolazos y
    valores <= 0). Usa `limpiar=False` si quieres cargar el archivo tal cual.

    Ejemplo:
        grafo = cargar_grafo("data/sample/transferencias.csv")
    """
    opciones.setdefault("limpiar", limpiar)
    grafo = Grafo().cargar(ruta, max_filas=max_filas, **opciones)
    if etiquetas:
        grafo.cargar_etiquetas(etiquetas)
    return grafo

## 1. Conectar con Google Drive

Esta celda monta Google Drive para poder leer el dataset desde
`Mi unidad/ED2/`. En local se salta sola.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive no disponible (normal si estás en local):", exc)

## Cargar el dataset

Sube **un solo archivo**:

```
transferencias.parquet
```

Puedes dejarlo en cualquiera de estas rutas:

```
/content/transferencias.parquet
Mi unidad/ED2/transferencias.parquet
data/procesado/transferencias.parquet
```

La celda busca el archivo, genera una muestra de USDC y construye el grafo
que usan las Tareas 1–4.

> Si estás en Colab, también puedes subirlo con **Archivo → Subir** o con:
>
> ```python
> # from google.colab import files
> # files.upload()
> ```

In [ ]:
import os
from collections import defaultdict, deque

CARPETA_DRIVE = "/content/drive/MyDrive/ED2"
CARPETA_LOCAL = "data"

USDC = "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"
CERO = "0x0000000000000000000000000000000000000000"


def buscar_archivo(*rutas):
    """Devuelve la primera ruta que exista, o None si no existe ninguna."""
    for ruta in rutas:
        if ruta and os.path.isfile(ruta):
            return ruta
    return None


RUTA_PARQUET = buscar_archivo(
    "/content/transferencias.parquet",
    f"{CARPETA_DRIVE}/transferencias.parquet",
    f"{CARPETA_LOCAL}/procesado/transferencias.parquet",
)

print("Parquet encontrado:", RUTA_PARQUET)

if RUTA_PARQUET is None:
    raise FileNotFoundError(
        "No encontré transferencias.parquet. Súbelo a /content/ o a "
        "Mi unidad/ED2/transferencias.parquet."
    )


def componente_mayor(aristas):
    """Conserva solo las aristas de la componente conexa más grande."""
    adyacencia = defaultdict(set)
    for origen, destino, _ in aristas:
        adyacencia[origen].add(destino)
        adyacencia[destino].add(origen)

    visitados = set()
    mejor = set()

    for inicio in adyacencia:
        if inicio in visitados:
            continue

        componente = set()
        cola = deque([inicio])
        visitados.add(inicio)

        while cola:
            nodo = cola.popleft()
            componente.add(nodo)
            for vecino in adyacencia[nodo]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    cola.append(vecino)

        if len(componente) > len(mejor):
            mejor = componente

    return [
        (origen, destino, peso)
        for origen, destino, peso in aristas
        if origen in mejor and destino in mejor
    ]


def muestra_desde_parquet(ruta):
    """Genera una muestra de USDC desde el Parquet."""
    try:
        import duckdb
    except ImportError:
        import subprocess
        import sys
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"],
            check=True,
        )
        import duckdb

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")

    filas = con.execute(
        """
        SELECT from_address, to_address, value
        FROM read_parquet(?)
        WHERE token_address = ?
        USING SAMPLE reservoir(65000 ROWS) REPEATABLE (42)
        """,
        [ruta, USDC],
    ).fetchall()

    aristas = []
    for origen, destino, valor in filas:
        origen = str(origen).lower()
        destino = str(destino).lower()
        valor = float(valor) / 1e6

        if not origen or not destino or origen == destino:
            continue
        if origen == CERO or destino == CERO:
            continue
        if valor <= 0:
            continue

        aristas.append((origen, destino, valor))

    return componente_mayor(aristas)


aristas = muestra_desde_parquet(RUTA_PARQUET)

grafo = Grafo()
for origen, destino, peso in aristas:
    grafo._agregar_arista(origen, destino, peso, limpiar=False)

print("Muestra generada desde el Parquet:")
print("  nodos   :", len(grafo.nodos))
print("  aristas :", len(grafo.aristas))
print("  resumen :", grafo.resumen())

### Etiquetas de exchanges

El Parquet no trae nombres, solo direcciones. Estas 296 etiquetas son metadatos
del dataset original y permiten que Louvain y PageRank muestren nombres como
`Binance 1`, `FTX Exchange` o `1inch.exchange` en lugar de direcciones.

Las etiquetas son parciales: no todas las direcciones tienen nombre.

In [ ]:
# Metadatos del dataset original: dirección -> nombre y tipo.
ETIQUETAS = {
    "0x11111254369792b2ca5d084ab5eea397ca8fa48b": {
        "nombre": "1inch.exchange",
        "tipo": "dex"
    },
    "0xe8fff15bb5e14095bfdfa8bb85d83cc900c23c56": {
        "nombre": "AfroDex",
        "tipo": "dex"
    },
    "0x4572f2554421bd64bef1c22c8a81840e8d496bea": {
        "nombre": "AirSwap",
        "tipo": "dex"
    },
    "0xdc1882f350b42ac9a23508996254b1915c78b204": {
        "nombre": "Allbit 1",
        "tipo": "dex"
    },
    "0xff6b1cdfd2d3e37977d7938aa06b6d89d6675e27": {
        "nombre": "Allbit 2",
        "tipo": "dex"
    },
    "0x0e8ba001a821f3ce0734763d008c9d7c957f5852": {
        "nombre": "AmadeusRelay",
        "tipo": "dex"
    },
    "0xc898fbee1cc94c0ff077faa5449915a506eff384": {
        "nombre": "Bamboo Relay",
        "tipo": "dex"
    },
    "0x3c020e014069df790d4f4e63fd297ba4e1c8e51f": {
        "nombre": "Bitcratic",
        "tipo": "dex"
    },
    "0x39fbd1140cd1fc298f00c3ea64b3591de94c67e7": {
        "nombre": "BitEye 1",
        "tipo": "dex"
    },
    "0xb5adb233f28c86cef693451b67e1f2d41da97d21": {
        "nombre": "Bitox.io",
        "tipo": "dex"
    },
    "0xe269e891a2ec8585a378882ffa531141205e92e9": {
        "nombre": "DDEX",
        "tipo": "dex"
    },
    "0x49497a4d914ae91d34ce80030fe620687bf333fd": {
        "nombre": "DDEX 1.0",
        "tipo": "dex"
    },
    "0xaf8ae6955d07776ab690e565ba6fbc79b8de3a5d": {
        "nombre": "DeversiFi",
        "tipo": "dex"
    },
    "0x745daa146934b27e3f0b6bff1a6e36b9b90fb131": {
        "nombre": "DEX.AG: Proxy Contract",
        "tipo": "dex"
    },
    "0x000000000000541e251335090ac5b47176af4f7e": {
        "nombre": "dex.blue",
        "tipo": "dex"
    },
    "0x7600977eb9effa627d6bd0da2e5be35e11566341": {
        "nombre": "DEx.top",
        "tipo": "dex"
    },
    "0x268be5667d4e5bd036bd608d0601ef1441604429": {
        "nombre": "DINNGO: Proxy",
        "tipo": "dex"
    },
    "0x58a5959a6c528c5d5e03f7b9e5102350e24005f1": {
        "nombre": "ERC dEX",
        "tipo": "dex"
    },
    "0x4aea7cf559f67cedcad07e12ae6bc00f07e8cf65": {
        "nombre": "EtherDelta 1",
        "tipo": "dex"
    },
    "0x8d12a197cb00d4747a1fe03395095ce2a5cc6819": {
        "nombre": "EtherDelta 2",
        "tipo": "dex"
    },
    "0xa5cc679a3528956e8032df4f03756c077c1ee3f4": {
        "nombre": "EtherMium",
        "tipo": "dex"
    },
    "0x2a0c0dbecc7e4d658f48e01e3fa353f44050c208": {
        "nombre": "IDEX 1",
        "tipo": "dex"
    },
    "0xa7a7899d944fe658c4b0a1803bab2f490bd3849e": {
        "nombre": "IDEX 2",
        "tipo": "dex"
    },
    "0x2cc42d1cd65af27cc999e41ef93d1a763dc821f8": {
        "nombre": "IDT Exchange",
        "tipo": "dex"
    },
    "0x04f062809b244e37e7fdc21d9409469c989c2342": {
        "nombre": "Joyso",
        "tipo": "dex"
    },
    "0x05f51aab068caa6ab7eeb672f88c180f67f17ec7": {
        "nombre": "ABCC",
        "tipo": "cex"
    },
    "0x2ddd202174a72514ed522e77972b461b03155525": {
        "nombre": "Alcumex Exchange",
        "tipo": "cex"
    },
    "0xf0c80fb9fb22bef8269cb6feb9a51130288a671f": {
        "nombre": "Artis Turba Exchange 1",
        "tipo": "cex"
    },
    "0x94597850916a49b3b152ee374e97260b99249f5b": {
        "nombre": "Artis Turba Exchange 2",
        "tipo": "cex"
    },
    "0x4df5f3610e2471095a130d7d934d551f3dde01ed": {
        "nombre": "ATAIX",
        "tipo": "cex"
    },
    "0xadb72986ead16bdbc99208086bd431c1aa38938e": {
        "nombre": "Beaxy",
        "tipo": "cex"
    },
    "0x7a10ec7d68a048bdae36a70e93532d31423170fa": {
        "nombre": "Bgogo 1",
        "tipo": "cex"
    },
    "0xce1bf8e51f8b39e51c6184e059786d1c0eaf360f": {
        "nombre": "Bgogo 2",
        "tipo": "cex"
    },
    "0xf73c3c65bde10bf26c2e1763104e609a41702efe": {
        "nombre": "Bibox",
        "tipo": "cex"
    },
    "0x0bb5de248dbbd31ee6c402c3c4a70293024acf74": {
        "nombre": "Bidesk 2",
        "tipo": "cex"
    },
    "0xed5cdb0d02152046e6f234ad578613831b9184d4": {
        "nombre": "Bidesk 3",
        "tipo": "cex"
    },
    "0xa30d8157911ef23c46c0eb71889efe6a648a41f7": {
        "nombre": "BigONE",
        "tipo": "cex"
    },
    "0x6eff3372fa352b239bb24ff91b423a572347000d": {
        "nombre": "BIKI.com 1",
        "tipo": "cex"
    },
    "0xf7793d27a1b76cdf14db7c83e82c772cf7c92910": {
        "nombre": "Bilaxy 1",
        "tipo": "cex"
    },
    "0xcce8d59affdd93be338fc77fa0a298c2cb65da59": {
        "nombre": "Bilaxy 2",
        "tipo": "cex"
    },
    "0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be": {
        "nombre": "Binance 1",
        "tipo": "cex"
    },
    "0xd551234ae421e3bcba99a0da6d736074f22192ff": {
        "nombre": "Binance 2",
        "tipo": "cex"
    },
    "0x564286362092d8e7936f0549571a803b203aaced": {
        "nombre": "Binance 3",
        "tipo": "cex"
    },
    "0x0681d8db095565fe8a346fa0277bffde9c0edbbf": {
        "nombre": "Binance 4",
        "tipo": "cex"
    },
    "0xfe9e8709d3215310075d67e3ed32a380ccf451c8": {
        "nombre": "Binance 5",
        "tipo": "cex"
    },
    "0x4e9ce36e442e55ecd9025b9a6e0d88485d628a67": {
        "nombre": "Binance 6",
        "tipo": "cex"
    },
    "0xbe0eb53f46cd790cd13851d5eff43d12404d33e8": {
        "nombre": "Binance 7",
        "tipo": "cex"
    },
    "0xf977814e90da44bfa03b6295a0616a897441acec": {
        "nombre": "Binance 8",
        "tipo": "cex"
    },
    "0x001866ae5b3de6caa5a51543fd9fb64f524f5478": {
        "nombre": "Binance 9",
        "tipo": "cex"
    },
    "0xab83d182f3485cf1d6ccdd34c7cfef95b4c08da4": {
        "nombre": "Binance JEX",
        "tipo": "cex"
    },
    "0x4b729cf402cfcffd057e254924b32241aedc1795": {
        "nombre": "Bit-Z",
        "tipo": "cex"
    },
    "0x7c49e1c0e33f3efb57d64b7690fa287c8d15b90a": {
        "nombre": "Bit2C",
        "tipo": "cex"
    },
    "0x5d375281582791a38e0348915fa9cbc6139e9c2a": {
        "nombre": "BitBlinx",
        "tipo": "cex"
    },
    "0x28ebe764b8f9a853509840645216d3c2c0fd774b": {
        "nombre": "BiteBTC",
        "tipo": "cex"
    },
    "0x1151314c646ce4e0efd76d1af4760ae66a9fe30f": {
        "nombre": "Bitfinex 1",
        "tipo": "cex"
    },
    "0x742d35cc6634c0532925a3b844bc454e4438f44e": {
        "nombre": "Bitfinex 2",
        "tipo": "cex"
    },
    "0x876eabf441b2ee5b5b0554fd502a8e0600950cfa": {
        "nombre": "Bitfinex 3",
        "tipo": "cex"
    },
    "0xab7c74abc0c4d48d1bdad5dcb26153fc8780f83e": {
        "nombre": "Bitfinex: MultiSig 1",
        "tipo": "cex"
    },
    "0xc6cde7c39eb2f0f0095f41570af89efc2c1ea828": {
        "nombre": "Bitfinex: MultiSig 2",
        "tipo": "cex"
    },
    "0xdf5021a4c1401f1125cd347e394d977630e17cf7": {
        "nombre": "BITFRONT",
        "tipo": "cex"
    },
    "0x88d34944cf554e9cccf4a24292d891f620e9c94f": {
        "nombre": "Bithumb 1",
        "tipo": "cex"
    },
    "0x186549a4ae594fc1f70ba4cffdac714b405be3f9": {
        "nombre": "Bithumb 10",
        "tipo": "cex"
    },
    "0xd273bd546b11bd60214a2f9d71f22a088aafe31b": {
        "nombre": "Bithumb 11",
        "tipo": "cex"
    },
    "0x3052cd6bf951449a984fe4b5a38b46aef9455c8e": {
        "nombre": "Bithumb 2",
        "tipo": "cex"
    },
    "0x2140efd7ba31169c69dfff6cdc66c542f0211825": {
        "nombre": "Bithumb 3",
        "tipo": "cex"
    },
    "0xa0ff1e0f30b5dda2dc01e7e828290bc72b71e57d": {
        "nombre": "Bithumb 4",
        "tipo": "cex"
    },
    "0xc1da8f69e4881efe341600620268934ef01a3e63": {
        "nombre": "Bithumb 5",
        "tipo": "cex"
    },
    "0xb4460b75254ce0563bb68ec219208344c7ea838c": {
        "nombre": "Bithumb 6",
        "tipo": "cex"
    },
    "0x15878e87c685f866edfaf454be6dc06fa517b35b": {
        "nombre": "Bithumb 7",
        "tipo": "cex"
    },
    "0x31d03f07178bcd74f9099afebd23b0ae30184ab5": {
        "nombre": "Bithumb 8",
        "tipo": "cex"
    },
    "0xed48dc0628789c2956b1e41726d062a86ec45bff": {
        "nombre": "Bithumb 9",
        "tipo": "cex"
    },
    "0x3fbe1f8fc5ddb27d428aa60f661eaaab0d2000ce": {
        "nombre": "Bithumb: Contract 1",
        "tipo": "cex"
    },
    "0xe79eef9b9388a4ff70ed7ec5bccd5b928ebb8bd1": {
        "nombre": "BitMart",
        "tipo": "cex"
    },
    "0x03bdf69b1322d623836afbd27679a1c0afa067e9": {
        "nombre": "Bitmax 1",
        "tipo": "cex"
    },
    "0x4b1a99467a284cc690e3237bc69105956816f762": {
        "nombre": "Bitmax 2",
        "tipo": "cex"
    },
    "0x4524baa98f9a3b9dec57caae7633936ef96bd708": {
        "nombre": "LedgerDex",
        "tipo": "dex"
    },
    "0xc692453625023c6e03fec04158ea31ab4de2650a": {
        "nombre": "LocalCoin Dex 1",
        "tipo": "dex"
    },
    "0x37c4bcaba4bcf3a605414236b8b108f160eb45a6": {
        "nombre": "LocalCoin Dex 2",
        "tipo": "dex"
    },
    "0x8d1c1571367a148e92d6ac83494b1bdf3b497d07": {
        "nombre": "LocalCoin DEX 3",
        "tipo": "dex"
    },
    "0x944644ea989ec64c2ab9ef341d383cef586a5777": {
        "nombre": "LoopringDEX: Beta 1",
        "tipo": "dex"
    },
    "0x51a2b1a38ec83b56009d5e28e6222dbb56c23c22": {
        "nombre": "nDEX Market",
        "tipo": "dex"
    },
    "0x794e6e91555438afc3ccf1c5076a74f42133d08d": {
        "nombre": "OasisDEX",
        "tipo": "dex"
    },
    "0xc22d5b2951db72b44cfb8089bb8cd374a3c354ea": {
        "nombre": "OpenRelay",
        "tipo": "dex"
    },
    "0xd2045edc40199019e221d71c0913343f7908d0d5": {
        "nombre": "Paradex",
        "tipo": "dex"
    },
    "0xf92c1ad75005e6436b4ee84e88cb23ed8a290988": {
        "nombre": "ParaSwap",
        "tipo": "dex"
    },
    "0xa258b39954cef5cb142fd567a46cddb31a670124": {
        "nombre": "Radar Relay",
        "tipo": "dex"
    },
    "0x1f0d1de1558582ad6f13763f477119a1455502af": {
        "nombre": "Saturn Network 1",
        "tipo": "dex"
    },
    "0xaa5bbd5a177a588b9f213505ca3740b444dbd586": {
        "nombre": "Saturn Network 2",
        "tipo": "dex"
    },
    "0x55890b06f0877a01bb5349d93b202961f8e27a9b": {
        "nombre": "Shark Relay",
        "tipo": "dex"
    },
    "0x9a2d163ab40f88c625fd475e807bbc3556566f80": {
        "nombre": "SingularX",
        "tipo": "dex"
    },
    "0x0681e844593a051e2882ec897ecd5444efe19ff2": {
        "nombre": "Star Bit Ex",
        "tipo": "dex"
    },
    "0x7ee7ca6e75de79e618e88bdf80d0b1db136b22d0": {
        "nombre": "Switcheo Exchange V2",
        "tipo": "dex"
    },
    "0x7219612be7036d1bfa933e16ca1246008f38c5fe": {
        "nombre": "The Ocean",
        "tipo": "dex"
    },
    "0x1ce7ae555139c5ef5a57cc8d814a867ee6ee33d8": {
        "nombre": "Token.Store",
        "tipo": "dex"
    },
    "0x5e150a33ffa97a8d22f59c77ae5487b089ef62e9": {
        "nombre": "TokenJar",
        "tipo": "dex"
    },
    "0xdc6c91b569c98f9f6f74d90f9beff99fdaf4248b": {
        "nombre": "Tokenlon",
        "tipo": "dex"
    },
    "0x77208a6000691e440026bed1b178ef4661d37426": {
        "nombre": "Totle: Primary",
        "tipo": "dex"
    },
    "0xbd2a43799b83d9d0ff56b85d4c140bce3d1d1c6c": {
        "nombre": "UniswapEX",
        "tipo": "dex"
    },
    "0xd97d09f3bd931a14382ac60f156c1285a56bb51b": {
        "nombre": "WeDEX: Beta 2",
        "tipo": "dex"
    },
    "0x986a2fca9eda0e06fbf7839b89bfc006ee2a23dd": {
        "nombre": "Bitmax 3",
        "tipo": "cex"
    },
    "0x00bdb5699745f5b860228c8f939abf1b9ae374ed": {
        "nombre": "Bitstamp 1",
        "tipo": "cex"
    },
    "0x1522900b6dafac587d499a862861c0869be6e428": {
        "nombre": "Bitstamp 2",
        "tipo": "cex"
    },
    "0x9a9bed3eb03e386d66f8a29dc67dc29bbb1ccb72": {
        "nombre": "Bitstamp 3",
        "tipo": "cex"
    },
    "0x059799f2261d37b829c2850cee67b5b975432271": {
        "nombre": "Bitstamp 4",
        "tipo": "cex"
    },
    "0x4c766def136f59f6494f0969b1355882080cf8e0": {
        "nombre": "Bitstamp 5",
        "tipo": "cex"
    },
    "0xc5b611f502a0dcf6c3188fd494061ae29b2baa4f": {
        "nombre": "Bitstamp 6",
        "tipo": "cex"
    },
    "0x1b8a38ea02ceda9440e00c1aeba26ee2dc570423": {
        "nombre": "BITStorage",
        "tipo": "cex"
    },
    "0xfbb1b73c4f0bda4f67dca266ce6ef42f520fbb98": {
        "nombre": "Bittrex 1",
        "tipo": "cex"
    },
    "0xe94b04a0fed112f3664e45adb2b8915693dd5ff3": {
        "nombre": "Bittrex 2",
        "tipo": "cex"
    },
    "0x66f820a414680b5bcda5eeca5dea238543f42054": {
        "nombre": "Bittrex 3",
        "tipo": "cex"
    },
    "0xaa90b4aae74cee41e004bc45e45a427406c4dcae": {
        "nombre": "BitUN.io 1",
        "tipo": "cex"
    },
    "0xf8d04a720520d0bcbc722b1d21ca194aa22699f2": {
        "nombre": "BitUN.io 2",
        "tipo": "cex"
    },
    "0xfb9f7f41319157ac5c5dccae308a63a4337ad5d9": {
        "nombre": "Bity.com",
        "tipo": "cex"
    },
    "0x00cdc153aa8894d08207719fe921fff964f28ba3": {
        "nombre": "Bitzlato",
        "tipo": "cex"
    },
    "0x007174732705604bbbf77038332dc52fd5a5000c": {
        "nombre": "BlockTrades Exchange",
        "tipo": "cex"
    },
    "0x1c00d840ccaa67c494109f46e55cfeb2d8562f5c": {
        "nombre": "BTC-Alpha Exchange",
        "tipo": "cex"
    },
    "0x73957709695e73fd175582105c44743cf0fb6f2f": {
        "nombre": "Bw.com",
        "tipo": "cex"
    },
    "0xd7c866d0d536937bf9123e02f7c052446588189f": {
        "nombre": "C2CX: Hot Wallet",
        "tipo": "cex"
    },
    "0x4dc98c79a52968a6c20ce9a7a08d5e8d1c2d5605": {
        "nombre": "CamboChanger",
        "tipo": "cex"
    },
    "0x88988d6ef12d7084e34814b9edafa01ae0d05082": {
        "nombre": "CamboChanger 2",
        "tipo": "cex"
    },
    "0x72bcfa6932feacd91cb2ea44b0731ed8ae04d0d3": {
        "nombre": "Cashierest",
        "tipo": "cex"
    },
    "0x7a56f645dcb513d0326cbaa048e9106ff6d4cd5f": {
        "nombre": "Catex Exchange",
        "tipo": "cex"
    },
    "0xfd648cc72f1b4e71cbdda7a0a91fe34d32abd656": {
        "nombre": "ChainX",
        "tipo": "cex"
    },
    "0x96fc4553a00c117c5b0bed950dd625d1c16dc894": {
        "nombre": "Changelly",
        "tipo": "cex"
    },
    "0x8958618332df62af93053cb9c535e26462c959b0": {
        "nombre": "Cobinhood 1",
        "tipo": "cex"
    },
    "0xb726da4fbdc3e4dbda97bb20998cf899b0e727e0": {
        "nombre": "Cobinhood 2",
        "tipo": "cex"
    },
    "0x9539e0b14021a43cde41d9d45dc34969be9c7cb0": {
        "nombre": "Coinbene",
        "tipo": "cex"
    },
    "0x33683b94334eebc9bd3ea85ddbda4a86fb461405": {
        "nombre": "Coinbene: Cold Wallet",
        "tipo": "cex"
    },
    "0xb6ba1931e4e74fd080587688f6db10e830f810d5": {
        "nombre": "Coindelta",
        "tipo": "cex"
    },
    "0xb9ee1e551f538a464e8f8c41e9904498505b49b0": {
        "nombre": "Coinex",
        "tipo": "cex"
    },
    "0x4b01721f0244e7c5b5f63c20942850e447f5a5ee": {
        "nombre": "CoinExchange.io",
        "tipo": "cex"
    },
    "0xd4bddf5e3d0435d7a6214a0b949c7bb58621f37c": {
        "nombre": "Coinhako",
        "tipo": "cex"
    },
    "0xf2d4766ad705e3a5c9ba5b0436b473085f82f82f": {
        "nombre": "Coinhako: Warm Wallet",
        "tipo": "cex"
    },
    "0xa270f3ad1a7a82e6a3157f12a900f1e25bc4fbfd": {
        "nombre": "CoinMetro Exchange",
        "tipo": "cex"
    },
    "0x167a9333bf582556f35bd4d16a7e80e191aa6476": {
        "nombre": "Coinone 1",
        "tipo": "cex"
    },
    "0x1e2fcfd26d36183f1a5d90f0e6296915b02bcb40": {
        "nombre": "Coinone 2",
        "tipo": "cex"
    },
    "0xd0808da05cc71a9f308d330bc9c5c81bbc26fc59": {
        "nombre": "Coinswitch",
        "tipo": "cex"
    },
    "0x0d6b5a54f940bf3d52e438cab785981aaefdf40c": {
        "nombre": "COSS.io 1",
        "tipo": "cex"
    },
    "0xd1560b3984b7481cd9a8f40435a53c860187174d": {
        "nombre": "COSS.io 2",
        "tipo": "cex"
    },
    "0x521db06bf657ed1d6c98553a70319a8ddbac75a3": {
        "nombre": "CREX24",
        "tipo": "cex"
    },
    "0x5baeac0a0417a05733884852aa068b706967e790": {
        "nombre": "Cryptopia 1",
        "tipo": "cex"
    },
    "0x2984581ece53a4390d1f568673cf693139c97049": {
        "nombre": "Cryptopia 2",
        "tipo": "cex"
    },
    "0xe17ee7b3c676701c66b395a35f0df4c2276a344e": {
        "nombre": "DigiFinex",
        "tipo": "cex"
    },
    "0x2101e480e22c953b37b9d0fe6551c1354fe705e6": {
        "nombre": "DMEX",
        "tipo": "cex"
    },
    "0xf1c525a488a848b58b95d79da48c21ce434290f7": {
        "nombre": "Eidoo",
        "tipo": "cex"
    },
    "0x8d76166c22658a144c0211d87abf152e6a2d9d95": {
        "nombre": "Eterbase",
        "tipo": "cex"
    },
    "0xd3808c5d48903be1490989f3fce2a2b3890e8eb6": {
        "nombre": "Exchange A",
        "tipo": "cex"
    },
    "0x1fd6267f0d86f62d88172b998390afee2a1f54b6": {
        "nombre": "Exmo 1",
        "tipo": "cex"
    },
    "0xd7b9a9b2f665849c4071ad5af77d8c76aa30fb32": {
        "nombre": "Exmo 2",
        "tipo": "cex"
    },
    "0x94fe3ad91dacba8ec4b82f56ff7c122181f1535d": {
        "nombre": "Faa.st",
        "tipo": "cex"
    },
    "0x915d7915f2b469bb654a7d903a5d4417cb8ea7df": {
        "nombre": "FCoin",
        "tipo": "cex"
    },
    "0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f": {
        "nombre": "FixedFloat",
        "tipo": "cex"
    },
    "0x91e18ee76483fa2ec5cfe2959df46673c2565be0": {
        "nombre": "Flybit",
        "tipo": "cex"
    },
    "0x0021845f4c2604c58f9ba5b7bff58d16a2ab372c": {
        "nombre": "Folgory Exchange",
        "tipo": "cex"
    },
    "0x2faf487a4414fe77e2327f0bf4ae2a264a776ad2": {
        "nombre": "FTX Exchange",
        "tipo": "cex"
    },
    "0x0d0707963952f2fba59dd06f2b425ace40b492fe": {
        "nombre": "Gate.io 1",
        "tipo": "cex"
    },
    "0x7793cd85c11a924478d358d49b05b37e91b5810f": {
        "nombre": "Gate.io 2",
        "tipo": "cex"
    },
    "0x1c4b70a3968436b9a0a9cf5205c787eb81bb558c": {
        "nombre": "Gate.io 3",
        "tipo": "cex"
    },
    "0xd793281182a0e3e023116004778f45c29fc14f19": {
        "nombre": "Gate.io: Contract 1",
        "tipo": "cex"
    },
    "0x9f5ca0012b9b72e8f3db57092a6f26bf4f13dc69": {
        "nombre": "GBX",
        "tipo": "cex"
    },
    "0xd24400ae8bfebb18ca49be86258a3c749cf46853": {
        "nombre": "Gemini 1",
        "tipo": "cex"
    },
    "0x6fc82a5fe25a5cdb58bc74600a40a69c065263f8": {
        "nombre": "Gemini 2",
        "tipo": "cex"
    },
    "0x61edcdf5bb737adffe5043706e7c5bb1f1a56eea": {
        "nombre": "Gemini 3",
        "tipo": "cex"
    },
    "0x07ee55aa48bb72dcc6e9d78256648910de513eca": {
        "nombre": "Gemini: Contract 1",
        "tipo": "cex"
    },
    "0x9fb01a2584aac5aae3fab1ed25f86c5269b32999": {
        "nombre": "GGBTC.com",
        "tipo": "cex"
    },
    "0x9c67e141c0472115aa1b98bd0088418be68fd249": {
        "nombre": "HitBTC 1",
        "tipo": "cex"
    },
    "0x59a5208b32e627891c389ebafc644145224006e8": {
        "nombre": "HitBTC 2",
        "tipo": "cex"
    },
    "0xa12431d0b9db640034b0cdfceef9cce161e62be4": {
        "nombre": "HitBTC 3",
        "tipo": "cex"
    },
    "0x980a4732c8855ffc8112e6746bd62095b4c2228f": {
        "nombre": "Hoo.com",
        "tipo": "cex"
    },
    "0x274f3c32c90517975e29dfc209a23f315c1e5fc7": {
        "nombre": "Hotbit 1",
        "tipo": "cex"
    },
    "0x8533a0bd9310eb63e7cc8e1116c18a3d67b1976a": {
        "nombre": "Hotbit 2",
        "tipo": "cex"
    },
    "0xab5c66752a9e8167967685f1450532fb96d5d24f": {
        "nombre": "Huobi 1",
        "tipo": "cex"
    },
    "0xe93381fb4c4f14bda253907b18fad305d799241a": {
        "nombre": "Huobi 10",
        "tipo": "cex"
    },
    "0xfa4b5be3f2f84f56703c42eb22142744e95a2c58": {
        "nombre": "Huobi 11",
        "tipo": "cex"
    },
    "0x46705dfff24256421a05d056c29e81bdc09723b8": {
        "nombre": "Huobi 12",
        "tipo": "cex"
    },
    "0x32598293906b5b17c27d657db3ad2c9b3f3e4265": {
        "nombre": "Huobi 13",
        "tipo": "cex"
    },
    "0x5861b8446a2f6e19a067874c133f04c578928727": {
        "nombre": "Huobi 14",
        "tipo": "cex"
    },
    "0x926fc576b7facf6ae2d08ee2d4734c134a743988": {
        "nombre": "Huobi 15",
        "tipo": "cex"
    },
    "0xeec606a66edb6f497662ea31b5eb1610da87ab5f": {
        "nombre": "Huobi 16",
        "tipo": "cex"
    },
    "0x7ef35bb398e0416b81b019fea395219b65c52164": {
        "nombre": "Huobi 17",
        "tipo": "cex"
    },
    "0x229b5c097f9b35009ca1321ad2034d4b3d5070f6": {
        "nombre": "Huobi 18",
        "tipo": "cex"
    },
    "0xd8a83b72377476d0a66683cde20a8aad0b628713": {
        "nombre": "Huobi 19",
        "tipo": "cex"
    },
    "0x6748f50f686bfbca6fe8ad62b22228b87f31ff2b": {
        "nombre": "Huobi 2",
        "tipo": "cex"
    },
    "0x90e9ddd9d8d5ae4e3763d0cf856c97594dea7325": {
        "nombre": "Huobi 20",
        "tipo": "cex"
    },
    "0x18916e1a2933cb349145a280473a5de8eb6630cb": {
        "nombre": "Huobi 21",
        "tipo": "cex"
    },
    "0x6f48a3e70f0251d1e83a989e62aaa2281a6d5380": {
        "nombre": "Huobi 22",
        "tipo": "cex"
    },
    "0xf056f435ba0cc4fcd2f1b17e3766549ffc404b94": {
        "nombre": "Huobi 23",
        "tipo": "cex"
    },
    "0x137ad9c4777e1d36e4b605e745e8f37b2b62e9c5": {
        "nombre": "Huobi 24",
        "tipo": "cex"
    },
    "0x5401dbf7da53e1c9dbf484e3d69505815f2f5e6e": {
        "nombre": "Huobi 25",
        "tipo": "cex"
    },
    "0x034f854b44d28e26386c1bc37ff9b20c6380b00d": {
        "nombre": "Huobi 26",
        "tipo": "cex"
    },
    "0x0577a79cfc63bbc0df38833ff4c4a3bf2095b404": {
        "nombre": "Huobi 27",
        "tipo": "cex"
    },
    "0x0c6c34cdd915845376fb5407e0895196c9dd4eec": {
        "nombre": "Huobi 28",
        "tipo": "cex"
    },
    "0x794d28ac31bcb136294761a556b68d2634094153": {
        "nombre": "Huobi 29",
        "tipo": "cex"
    },
    "0xfdb16996831753d5331ff813c29a93c76834a0ad": {
        "nombre": "Huobi 3",
        "tipo": "cex"
    },
    "0xfd54078badd5653571726c3370afb127351a6f26": {
        "nombre": "Huobi 30",
        "tipo": "cex"
    },
    "0xb4cd0386d2db86f30c1a11c2b8c4f4185c1dade9": {
        "nombre": "Huobi 31",
        "tipo": "cex"
    },
    "0x4d77a1144dc74f26838b69391a6d3b1e403d0990": {
        "nombre": "Huobi 32",
        "tipo": "cex"
    },
    "0x28ffe35688ffffd0659aee2e34778b0ae4e193ad": {
        "nombre": "Huobi 33",
        "tipo": "cex"
    },
    "0xcac725bef4f114f728cbcfd744a731c2a463c3fc": {
        "nombre": "Huobi 34",
        "tipo": "cex"
    },
    "0xeee28d484628d41a82d01e21d12e2e78d69920da": {
        "nombre": "Huobi 4",
        "tipo": "cex"
    },
    "0x5c985e89dde482efe97ea9f1950ad149eb73829b": {
        "nombre": "Huobi 5",
        "tipo": "cex"
    },
    "0xdc76cd25977e0a5ae17155770273ad58648900d3": {
        "nombre": "Huobi 6",
        "tipo": "cex"
    },
    "0xadb2b42f6bd96f5c65920b9ac88619dce4166f94": {
        "nombre": "Huobi 7",
        "tipo": "cex"
    },
    "0xa8660c8ffd6d578f657b72c0c811284aef0b735e": {
        "nombre": "Huobi 8",
        "tipo": "cex"
    },
    "0x1062a747393198f70f71ec65a582423dba7e5ab3": {
        "nombre": "Huobi 9",
        "tipo": "cex"
    },
    "0x3c11c3025ce387d76c2eddf1493ec55a8cc2a0f7": {
        "nombre": "Idax.global",
        "tipo": "cex"
    },
    "0x51836a753e344257b361519e948ffcaf5fb8d521": {
        "nombre": "Indodax 1",
        "tipo": "cex"
    },
    "0x9cbadd5ce7e14742f70414a6dcbd4e7bb8712719": {
        "nombre": "Indodax 2",
        "tipo": "cex"
    },
    "0xb1a34309af7f29b4195a6b589737f86e14597ddc": {
        "nombre": "IndoEx LTD",
        "tipo": "cex"
    },
    "0xf09ff5eb57a3c8880934201c91eefbd1ace72033": {
        "nombre": "JPMarket.cc",
        "tipo": "cex"
    },
    "0x2910543af39aba0cd09dbb2d50200b3e800a63d2": {
        "nombre": "Kraken 1",
        "tipo": "cex"
    },
    "0x0a869d79a7052c7f1b55a8ebabbea3420f0d1e13": {
        "nombre": "Kraken 2",
        "tipo": "cex"
    },
    "0xe853c56864a2ebe4576a807d26fdc4a0ada51919": {
        "nombre": "Kraken 3",
        "tipo": "cex"
    },
    "0x267be1c1d684f78cb4f6a176c4911b741e4ffdc0": {
        "nombre": "Kraken 4",
        "tipo": "cex"
    },
    "0xfa52274dd61e1643d2205169732f29114bc240b3": {
        "nombre": "Kraken 5",
        "tipo": "cex"
    },
    "0xe8a0e282e6a3e8023465accd47fae39dd5db010b": {
        "nombre": "Kryptono 1",
        "tipo": "cex"
    },
    "0x629a7144235259336ea2694167f3c8b856edd7dc": {
        "nombre": "Kryptono 2",
        "tipo": "cex"
    },
    "0x30b71d015f60e2f959743038ce0aaec9b4c1ea44": {
        "nombre": "Kryptono 3",
        "tipo": "cex"
    },
    "0x2b5634c42055806a59e9107ed44d43c426e58258": {
        "nombre": "KuCoin 1",
        "tipo": "cex"
    },
    "0x689c56aef474df92d44a1b70850f808488f9769c": {
        "nombre": "KuCoin 2",
        "tipo": "cex"
    },
    "0xea81ce54a0afa10a027f65503bd52fba83d745b8": {
        "nombre": "Kuna.io",
        "tipo": "cex"
    },
    "0x0861fca546225fbf8806986d211c8398f7457734": {
        "nombre": "LAToken 1",
        "tipo": "cex"
    },
    "0x7891b20c690605f4e370d6944c8a5dbfac5a451c": {
        "nombre": "LAToken 2",
        "tipo": "cex"
    },
    "0x1b6c1a0e20af81b922cb454c3e52408496ee7201": {
        "nombre": "LAToken 3",
        "tipo": "cex"
    },
    "0x8271b2e8cbe29396e9563229030c89679b9470db": {
        "nombre": "Liqui.io 1",
        "tipo": "cex"
    },
    "0x5e575279bf9f4acf0a130c186861454247394c06": {
        "nombre": "Liqui.io 2",
        "tipo": "cex"
    },
    "0xedbb72e6b3cf66a792bff7faac5ea769fe810517": {
        "nombre": "Liquid",
        "tipo": "cex"
    },
    "0x243bec9256c9a3469da22103891465b47583d9f1": {
        "nombre": "Livecoin.net",
        "tipo": "cex"
    },
    "0xe03c23519e18d64f144d2800e30e81b0065c48b5": {
        "nombre": "Mercatox",
        "tipo": "cex"
    },
    "0x75e89d5979e4f6fba9f97c104c2f0afb3f1dcb88": {
        "nombre": "MXC 1",
        "tipo": "cex"
    },
    "0x0211f3cedbef3143223d3acf0e589747933e8527": {
        "nombre": "MXC 2",
        "tipo": "cex"
    },
    "0xae7006588d03bd15d6954e3084a7e644596bc251": {
        "nombre": "NEXBIT Pro",
        "tipo": "cex"
    },
    "0x6cc5f688a315f3dc28a7781717a9a798a59fda7b": {
        "nombre": "OKEx 1",
        "tipo": "cex"
    },
    "0x236f9f97e0e62388479bf9e5ba4889e46b0273c3": {
        "nombre": "OKEx 2",
        "tipo": "cex"
    },
    "0xa7efae728d2936e78bda97dc267687568dd593f3": {
        "nombre": "OKEx 3",
        "tipo": "cex"
    },
    "0x03e3ff995863828554282e80870b489cc31dc8bc": {
        "nombre": "OMGFIN Exchange",
        "tipo": "cex"
    },
    "0xaeec6f5aca72f3a005af1b3420ab8c8c7009bac8": {
        "nombre": "OTCBTC",
        "tipo": "cex"
    },
    "0xbd8ef191caa1571e8ad4619ae894e07a75de0c35": {
        "nombre": "Paribu 1",
        "tipo": "cex"
    },
    "0x2bb97b6cf6ffe53576032c11711d59bd056830ee": {
        "nombre": "Paribu 2",
        "tipo": "cex"
    },
    "0xfb90501083a3b6af766c8da35d3dde01eb0d2a68": {
        "nombre": "Paribu 3",
        "tipo": "cex"
    },
    "0xabc74170f3cb8ab352820c39cc1d1e05ce9e41d3": {
        "nombre": "Paribu 4",
        "tipo": "cex"
    },
    "0xd4dcd2459bb78d7a645aa7e196857d421b10d93f": {
        "nombre": "Peatio",
        "tipo": "cex"
    },
    "0x32be343b94f860124dc4fee278fdcbd38c102d88": {
        "nombre": "Poloniex 1",
        "tipo": "cex"
    },
    "0x209c4784ab1e8183cf58ca33cb740efbf3fc18ef": {
        "nombre": "Poloniex 2",
        "tipo": "cex"
    },
    "0xb794f5ea0ba39494ce839613fffba74279579268": {
        "nombre": "Poloniex 3",
        "tipo": "cex"
    },
    "0xa910f92acdaf488fa6ef02174fb86208ad7722ba": {
        "nombre": "Poloniex 4",
        "tipo": "cex"
    },
    "0xaa9fa73dfe17ecaa2c89b39f0bb2779613c5fc3b": {
        "nombre": "Poloniex: BAT",
        "tipo": "cex"
    },
    "0x2fa2bc2ce6a4f92952921a4caa46b3727d24a1ec": {
        "nombre": "Poloniex: BNT",
        "tipo": "cex"
    },
    "0x31a2feb9b5d3b5f4e76c71d6c92fc46ebb3cb1c1": {
        "nombre": "Poloniex: CVC",
        "tipo": "cex"
    },
    "0x6b71834d65c5c4d8ed158d54b47e6ea4ff4e5437": {
        "nombre": "Poloniex: FOAM",
        "tipo": "cex"
    },
    "0x48d466b7c0d32b61e8a82cd2bcf060f7c3f966df": {
        "nombre": "Poloniex: GNO",
        "tipo": "cex"
    },
    "0x0536806df512d6cdde913cf95c9886f65b1d3462": {
        "nombre": "Poloniex: GNT",
        "tipo": "cex"
    },
    "0x8d451ae5ee8f557a9ce7a9d7be8a8cb40002d5cb": {
        "nombre": "Poloniex: KNC",
        "tipo": "cex"
    },
    "0xbd2ec7c608a06fe975dbdca729e84dedb34ecc21": {
        "nombre": "Poloniex: LOOM",
        "tipo": "cex"
    },
    "0xc0e30823e5e628df8bc9bf2636a347e1512f0ecb": {
        "nombre": "Poloniex: MANA",
        "tipo": "cex"
    },
    "0x65f9b2e4d7aaeb40ffea8c6f5844d5ad7da257e0": {
        "nombre": "Poloniex: NXC",
        "tipo": "cex"
    },
    "0x36b01066b7fa4a0fdb2968ea0256c848e9135674": {
        "nombre": "Poloniex: OMG",
        "tipo": "cex"
    },
    "0xab11204cfeaccffa63c2d23aef2ea9accdb0a0d5": {
        "nombre": "Poloniex: REP",
        "tipo": "cex"
    },
    "0x6795cf8eb25585eadc356ae32ac6641016c550f2": {
        "nombre": "Poloniex: SNT",
        "tipo": "cex"
    },
    "0xfbf2173154f7625713be22e0504404ebfe021eae": {
        "nombre": "Poloniex: STORJ",
        "tipo": "cex"
    },
    "0x6f803466bcd17f44fa18975bf7c509ba64bf3825": {
        "nombre": "Poloniex: USDC",
        "tipo": "cex"
    },
    "0xead6be34ce315940264519f250d8160f369fa5cd": {
        "nombre": "Poloniex: ZRX",
        "tipo": "cex"
    },
    "0xd344539efe31f8b6de983a0cab4fb721fc69c547": {
        "nombre": "QuantaEx 1",
        "tipo": "cex"
    },
    "0x5ca39c42f4dee3a5ba8fec3ad4902157d48700bf": {
        "nombre": "QuantaEx 2",
        "tipo": "cex"
    },
    "0x2a048d9a8ffdd239f063b09854976c3049ae659c": {
        "nombre": "QuantaEx 3",
        "tipo": "cex"
    },
    "0xb8cf411b956b3f9013c1d0ac8c909b086218207c": {
        "nombre": "Remitano 1",
        "tipo": "cex"
    },
    "0x2819c144d5946404c0516b6f817a960db37d4929": {
        "nombre": "Remitano 2",
        "tipo": "cex"
    },
    "0x120a270bbc009644e35f0bb6ab13f95b8199c4ad": {
        "nombre": "ShapeShift 1",
        "tipo": "cex"
    },
    "0x9e6316f44baeeee5d41a1070516cc5fa47baf227": {
        "nombre": "ShapeShift 2",
        "tipo": "cex"
    },
    "0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413": {
        "nombre": "ShapeShift 3",
        "tipo": "cex"
    },
    "0x563b377a956c80d77a7c613a9343699ad6123911": {
        "nombre": "ShapeShift 4",
        "tipo": "cex"
    },
    "0xd3273eba07248020bf98a8b560ec1576a612102f": {
        "nombre": "ShapeShift 5",
        "tipo": "cex"
    },
    "0x3b0bc51ab9de1e5b7b6e34e5b960285805c41736": {
        "nombre": "ShapeShift 6",
        "tipo": "cex"
    },
    "0xeed16856d551569d134530ee3967ec79995e2051": {
        "nombre": "ShapeShift 7",
        "tipo": "cex"
    },
    "0x324cc2c9fb379ea7a0d1c0862c3b48ca28d174a4": {
        "nombre": "SouthXchange",
        "tipo": "cex"
    },
    "0x91f6d99b232153cb655ad3e0d05e13ef505f6cd5": {
        "nombre": "Sparrow Exchange 1",
        "tipo": "cex"
    },
    "0xe855283086fbee485aecf2084345a91424c23954": {
        "nombre": "Sparrow Exchange 2",
        "tipo": "cex"
    },
    "0xa96b536eef496e21f5432fd258b6f78cf3673f74": {
        "nombre": "Switchain",
        "tipo": "cex"
    },
    "0xea3a46bd1dbd0620d80037f70d0bf7c7dc5a837c": {
        "nombre": "TAGZ 1",
        "tipo": "cex"
    },
    "0xed8204345a0cf4639d2db61a4877128fe5cf7599": {
        "nombre": "TAGZ 2",
        "tipo": "cex"
    },
    "0x3613ef1125a078ef96ffc898c4ec28d73c5b8c52": {
        "nombre": "Tidex 1",
        "tipo": "cex"
    },
    "0x0a73573cf2903d2d8305b1ecb9e9730186a312ae": {
        "nombre": "Tidex 2",
        "tipo": "cex"
    },
    "0x0068eb681ec52dbd9944517d785727310b494575": {
        "nombre": "Tokocrypto",
        "tipo": "cex"
    },
    "0xb2cc3cdd53fc9a1aeaf3a68edeba2736238ddc5d": {
        "nombre": "TopBTC",
        "tipo": "cex"
    },
    "0x1119aaefb02bf12b84d28a5d8ea48ec3c90ef1db": {
        "nombre": "Trade.io",
        "tipo": "cex"
    },
    "0x2f1233ec3a4930fd95874291db7da9e90dfb2f03": {
        "nombre": "Uex",
        "tipo": "cex"
    },
    "0x390de26d772d2e2005c6d1d24afc902bae37a4bb": {
        "nombre": "Upbit 1",
        "tipo": "cex"
    },
    "0xba826fec90cefdf6706858e5fbafcb27a290fbe0": {
        "nombre": "Upbit 2",
        "tipo": "cex"
    },
    "0x5e032243d507c743b061ef021e2ec7fcc6d3ab89": {
        "nombre": "Upbit 3",
        "tipo": "cex"
    },
    "0xb436c96c6de1f50a160ed307317c275424dbe4f2": {
        "nombre": "Vinex",
        "tipo": "cex"
    },
    "0xf5bec430576ff1b82e44ddb5a1c93f6f9d0884f3": {
        "nombre": "Yobit.net",
        "tipo": "cex"
    },
    "0x8f3ab2c3b651382b07a76653d2be9eb4b87e1630": {
        "nombre": "YOOBTC 1",
        "tipo": "cex"
    },
    "0xd94c9ff168dc6aebf9b6cc86deff54f3fb0afc33": {
        "nombre": "Yunbi 1",
        "tipo": "cex"
    },
    "0x42da8a05cb7ed9a43572b5ba1b8f82a0a6e263dc": {
        "nombre": "Yunbi 2",
        "tipo": "cex"
    },
    "0x700f6912e5753e91ea3fae877a2374a2db1245d7": {
        "nombre": "Yunbi 3",
        "tipo": "cex"
    },
    "0x60d0cc2ae15859f69bf74dadb8ae3bd58434976b": {
        "nombre": "ZB.com",
        "tipo": "cex"
    }
}

grafo.etiquetas.update(ETIQUETAS)

print("Etiquetas cargadas:", len(grafo.etiquetas))
print("Ejemplo:", grafo.nombre("0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be"))

## Tarea 1 — Bellman-Ford: camino de costo mínimo

**Responsable:** Pablo Álvarez

**Estructura de datos:** lista de adyacencia + diccionario de distancias.

El costo de un camino es la suma de los pesos de sus aristas. Bellman-Ford
relaja todas las aristas `|V|-1` veces y detecta ciclos negativos.

La implementación está en la celda siguiente y se prueba con la muestra.

In [ ]:
"""
bellman_ford.py
===============
Tarea 1 — Caminos de costo mínimo.

Problema: encontrar la ruta más "barata" entre dos direcciones del grafo,
donde el costo de una ruta es la suma de los pesos de sus aristas.

Regla del proyecto: NO se puede usar Dijkstra. Por eso usamos Bellman-Ford,
que además es capaz de detectar ciclos de peso negativo.

Idea del algoritmo:
    1. Empezamos con distancia 0 en el origen e "infinito" en el resto.
    2. Repetimos |V|-1 veces: por cada arista (u -> v, w), si
       dist[u] + w < dist[v], entonces actualizamos dist[v] = dist[u] + w.
       (a esto se le llama "relajar" una arista).
    3. Al terminar, dist[v] es la distancia mínima desde el origen a v.

Complejidad: O(|V| * |E|).
"""

from typing import Dict, List, Optional, Tuple

INF = float("inf")


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en una lista de adyacencia.

    La lista de aristas viene de la carga única del dataset. Cada módulo
    construye su propia estructura, así no depende de la de otro.
    """
    from collections import defaultdict

    adj: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    for origen, destino, peso in aristas:
        adj[origen].append((destino, float(peso)))
    return dict(adj)


def bellman_ford(
    adj: Dict[str, List[Tuple[str, float]]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Calcula las distancias mínimas desde `origen` a todos los nodos.

    Parámetros:
        adj: lista de adyacencia {nodo: [(vecino, peso), ...]}.
        origen: nodo de partida.

    Retorna:
        (distancias, predecesores):
            - distancias[v] = costo mínimo desde `origen` hasta v
              (INF si v es inalcanzable).
            - predecesores[v] = nodo anterior a v en el camino mínimo
              (None para el origen).
    """
    # 1. Reunimos todos los nodos del grafo (incluye los que no tienen
    #    aristas salientes pero sí entrantes).
    nodos = set(adj.keys())
    for vecinos in adj.values():
        for dst, _ in vecinos:
            nodos.add(dst)

    # 2. Inicializamos distancias y predecesores.
    dist = {v: INF for v in nodos}
    dist[origen] = 0.0
    prev = {v: None for v in nodos}

    # 3. Relajamos todas las aristas |V|-1 veces.
    for _ in range(len(nodos) - 1):
        cambio = False
        for u in adj:
            for v, w in adj[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    prev[v] = u
                    cambio = True
        if not cambio:
            # Ya no mejora nada: terminamos antes de tiempo.
            break

    # 4. (Opcional) Una pasada extra para detectar ciclos de peso negativo.
    #    Con pesos positivos nunca ocurre, pero es una ventaja de Bellman-Ford.
    for u in adj:
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                raise ValueError("Se detectó un ciclo de peso negativo.")

    return dist, prev


def reconstruir_camino(
    prev: Dict[str, Optional[str]],
    origen: str,
    destino: str,
) -> List[str]:
    """Reconstruye el camino mínimo desde `origen` hasta `destino`.

    Usa el diccionario de predecesores que devuelve `bellman_ford`.

    Retorna:
        Lista de nodos [origen, ..., destino], o [] si no hay camino.
    """
    camino: List[str] = []
    actual: Optional[str] = destino

    # Retrocedemos desde el destino hasta el origen siguiendo `prev`.
    while actual is not None:
        camino.append(actual)
        actual = prev.get(actual)

    camino.reverse()
    return camino if camino and camino[0] == origen else []


def bellman_ford_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Igual que `bellman_ford`, pero recibe la lista compartida de aristas."""
    return bellman_ford(construir_adyacencia(aristas), origen)


from collections import Counter

salidas = Counter(origen for origen, _, _ in grafo.aristas)
etiquetados = sorted(
    (n for n in salidas if n in grafo.etiquetas),
    key=salidas.get, reverse=True,
)
origen = etiquetados[0] if etiquetados else max(salidas, key=salidas.get)

distancias, previos = bellman_ford_desde_aristas(grafo.aristas, origen)
alcanzables = sorted(
    (d, nodo) for nodo, d in distancias.items()
    if d != float("inf") and nodo != origen
)
inalcanzables = sum(1 for d in distancias.values() if d == float("inf"))

print(f"Origen: {grafo.nombre(origen)}")
print(f"Nodos alcanzables: {len(alcanzables):,}")
print(f"Nodos inalcanzables: {inalcanzables:,}\n")
print("Los 10 destinos más baratos:")
for d, destino in alcanzables[:10]:
    camino = reconstruir_camino(previos, origen, destino)
    nombres = " -> ".join(grafo.nombre(n) for n in camino)
    print(f"  {grafo.nombre(destino):<16} costo={d:>14,.2f}  ({nombres})")

## Tarea 2 — Centralidad y PageRank

**Responsable:** Jhon Arciniegas

**Estructura de datos:** lista de adyacencia + diccionario de puntajes.

- **Grado**: número de conexiones de un nodo.
- **PageRank**: importancia según quién apunta a quién.

PageRank se calcula con iteración de potencias y factor de amortiguación
`d = 0.85`.

La gráfica del top 5 y las validaciones de esta sección fueron aportadas por
Jhon Arciniegas (T2).

In [ ]:
from typing import Dict, List, Optional, Set, Tuple


def centralidad_grado(adj, nodos):
    """Centralidad de grado: C(v) = (vecinos distintos de v) / (|V| - 1).

    Cuenta direcciones DISTINTAS con las que v se conecta (envia o recibe),
    sin importar cuántas transferencias haya entre ellas. Va de 0 a 1
    """
    n = len(nodos)
    if n <= 1:
        return {v: 0.0 for v in nodos}
    vecinos = {v: set() for v in nodos}
    for u, destinos in adj.items():
        for v, _ in destinos:
            if u != v and u in vecinos and v in vecinos:
                vecinos[u].add(v)
                vecinos[v].add(u)
    return {v: len(vecinos[v]) / (n - 1) for v in nodos}


def pagerank(
    adj: Dict[str, List[Tuple[str, float]]],
    nodos: Set[str],
    amortiguacion: float = 0.85,
    max_iter: int = 100,
    tol: float = 1e-6,
) -> Dict[str, float]:
    """Calcula PageRank con el método de iteración de potencias.

    parametros:
        adj: lista de adyacencia
        nodos: conjunto de todos los vertices
        amortiguacion: factor de amortiguación d (0.85 por defecto)
        max_iter: máximo número de iteraciones
        tol: tolerancia de convergencia

    Retorna:
        {nodo: score} con scores que suman aproximadamente 1
    """
    n = len(nodos)
    if n == 0:
        return {}

    lista = list(nodos)

    # Puntaje inicial: todos los nodos igual de importantes (1/N).
    pr = {v: 1.0 / n for v in lista}

    # Grado de salida de cada nodo.
    grado_salida = {v: len(adj.get(v, [])) for v in lista}

    # Lista de adyacencia INVERSA: quién apunta a cada nodo.
    # Esto hace el cálculo más directo y eficiente.
    entrantes = {v: [] for v in lista}
    for u, vecinos in adj.items():
        for v, _ in vecinos:
            if v in entrantes:
                entrantes[v].append(u)

    for _ in range(max_iter):
        base = (1.0 - amortiguacion) / n

        # Los nodos "sumidero" (sin aristas de salida) no reparten su
        # importancia a nadie. Para que los puntajes sigan sumando 1, su masa
        # se reparte por igual entre todos los nodos.
        masa_sumidero = sum(pr[v] for v in lista if grado_salida[v] == 0)

        nuevo = {}
        for v in lista:
            suma = 0.0
            for u in entrantes[v]:
                if grado_salida[u] > 0:
                    suma += pr[u] / grado_salida[u]
            nuevo[v] = base + amortiguacion * (suma + masa_sumidero / n)

        # Medimos cuánto cambió el vector para saber si ya convergió.
        delta = sum(abs(nuevo[v] - pr[v]) for v in lista)
        pr = nuevo
        if delta < tol:
            break

    return pr


def top_n(
    puntajes: Dict[str, float],
    n: int = 10,
    etiquetas: Optional[Dict[str, Dict[str, str]]] = None,
) -> List[Tuple[str, float, str]]:
    """devuelve los `n` nodos con mayor puntaje, ordenados de mayor a menor

    retorna:
        lista de tuplas (direccion, puntaje, nombre) para mayor orden
    """
    etiquetas = etiquetas or {}

    # Orden: mayor puntaje primero; si empatan, por direccion (alfabetico)
    ordenados = sorted(puntajes.items(), key=lambda par: (-par[1], par[0]))[:n]

    resultado = []
    for addr, score in ordenados:
        # Si la etiqueta no existe o su nombre esta vacio, usamos la direccion
        nombre = etiquetas.get(addr, {}).get("nombre") or addr
        resultado.append((addr, score, nombre))
    return resultado


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en lista de adyacencia."""
    adj: Dict[str, List[Tuple[str, float]]] = {}
    for origen, destino, peso in aristas:
        adj.setdefault(origen, []).append((destino, float(peso)))
    return adj


def nodos_de_aristas(aristas: List[Tuple[str, str, float]]) -> Set[str]:
    """Devuelve todos los nodos que aparecen en la lista de aristas."""
    nodos: Set[str] = set()
    for origen, destino, _ in aristas:
        nodos.add(origen)
        nodos.add(destino)
    return nodos


def centralidad_grado_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Dict[str, float]:
    """Centralidad de grado a partir de la lista compartida de aristas."""
    return centralidad_grado(construir_adyacencia(aristas), nodos_de_aristas(aristas))


def pagerank_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    **opciones,
) -> Dict[str, float]:
    """PageRank a partir de la lista compartida de aristas."""
    return pagerank(
        construir_adyacencia(aristas),
        nodos_de_aristas(aristas),
        **opciones,
    )


import matplotlib.pyplot as plt

cent_grado = centralidad_grado_desde_aristas(grafo.aristas)
pr = pagerank_desde_aristas(grafo.aristas)

print("Top 10 por grado:")
for _, score, nombre in top_n(cent_grado, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

print("\nTop 10 por PageRank:")
for _, score, nombre in top_n(pr, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

# Gráfica y validaciones aportadas por Jhon Arciniegas (T2).
top_pr = top_n(pr, n=5, etiquetas=grafo.etiquetas)
nombres_pr = [nombre for _, _, nombre in top_pr]
scores_pr = [score for _, score, _ in top_pr]

plt.figure(figsize=(7, 4))
plt.barh(nombres_pr[::-1], scores_pr[::-1], color="steelblue")
plt.xlabel("PageRank")
plt.title("Top 5 direcciones por PageRank")
plt.tight_layout()
plt.show()

# Validaciones de centralidad (mismas que tests/test_centralidad.py).
assert centralidad_grado({"A": []}, {"A"})["A"] == 0.0
assert pagerank({}, set()) == {}
assert top_n({"A": 0.5, "B": 0.3}, n=2)[0][0] == "A"
print("Validaciones de centralidad: OK")

## Tarea 3 — Louvain: detección de comunidades

**Responsable:** Emerson Ballesteros

### Antes del código: ¿qué es una comunidad?

Imagina cuatro direcciones:

```
Ana   <--> Beto     (se transfieren mucho)
Carla <--> Dani     (se transfieren mucho)
Beto  -->  Carla    (se transfieren poquito)
```

Hay dos grupos naturales: `{Ana, Beto}` y `{Carla, Dani}`. Louvain intenta
encontrar esos grupos automáticamente, sin que le digamos cuántos hay.

### ¿Qué mide la modularidad Q?

Q compara las aristas que quedan **dentro** de los grupos con las que
aparecerían si las conexiones fueran al azar. Si hay más conexiones internas
de las esperadas, Q sube.

$$Q = \sum_c \left[\frac{L_c}{m} - \left(\frac{K_c}{2m}\right)^2\right]$$

- `L_c`: peso de las aristas internas de la comunidad `c`.
- `K_c`: suma de grados de los nodos de `c`.
- `m`: peso total del grafo.
- `Q > 0.3` sugiere una estructura de comunidades clara.
- `Q = 0` significa que la partición no es mejor que el azar.

### Paso a paso del algoritmo

1. Cada nodo empieza en su propia comunidad.
2. Se recorre la lista de nodos en orden aleatorio.
3. Para cada nodo, se prueba moverlo a la comunidad de cada vecino.
4. Se acepta el movimiento solo si Q aumenta.
5. Cuando una pasada completa no cambia nada, el algoritmo termina.

### Cómo leer el código

| Función | Qué hace |
|---|---|
| `simetrizar` | convierte el grafo dirigido en no dirigido |
| `calcular_grados` | suma los pesos de las aristas de cada nodo |
| `modularidad` | calcula Q de una partición |
| `louvain` | hace las pasadas y mueve los nodos |
| `desde_aristas` | construye el grafo no dirigido desde `grafo.aristas` |
| `louvain_desde_aristas` | atajo: construye el grafo y ejecuta Louvain |
| `renumerar_comunidades` | cambia los IDs por 0, 1, 2, ... para leerlos mejor |

### Qué mirar en los resultados

- Los **IDs de comunidad son arbitrarios**: la comunidad 0 no es mejor que la 5.
- Lo importante es **Q**, el **tamaño** de las comunidades y **qué exchanges**
  quedan juntos.
- Louvain es **heurístico**: dos semillas pueden dar particiones parecidas
  pero no idénticas.

### Limitaciones y posibles mejoras

- Esta versión recalcula Q completo en cada prueba: es fácil de leer, pero lenta.
- No incluye la fase de agregación de Louvain.
- Puede sufrir el límite de resolución: comunidades muy pequeñas pueden pasar
  desapercibidas.
- Mejoras: calcular ΔQ incremental, agregar la fase de agregación, probar
  varias semillas y comparar con Label Propagation o `python-louvain`.

In [ ]:
"""
comunidades.py
==============
Tarea 3 — Detección de comunidades con Louvain (versión didáctica).

Responsable: Emerson Ballesteros.

¿Qué es una comunidad?
----------------------
Una comunidad es un grupo de nodos que interactúa mucho entre sí y poco con
el resto de la red. Por ejemplo, los exchanges centralizados y sus usuarios
suelen formar un grupo, y los DEX con sus usuarios otro.

¿Cómo se mide si una partición es buena?
----------------------------------------
Con la **modularidad Q**. La idea es comparar las aristas que quedan *dentro*
de los grupos con las que aparecerían si las conexiones fueran al azar:

    Q = (1 / 2m) * suma_{i,j} [ A_ij - k_i*k_j/(2m) ] * delta(c_i, c_j)

    - A_ij: peso de la arista entre i y j.
    - k_i: grado (suma de pesos) del nodo i.
    - m: peso total del grafo.
    - delta(c_i, c_j): 1 si i y j están en la misma comunidad, 0 si no.

Para no recorrer todos los pares de nodos, el código usa una fórmula
equivalente que se calcula por comunidad:

    Q = suma_c [ L_c / m - (K_c / (2m))^2 ]

    - L_c: peso de las aristas internas de la comunidad c.
    - K_c: suma de grados de los nodos de la comunidad c.

Valores de Q por encima de ~0.3 sugieren una buena estructura de comunidades.

Algoritmo (fase principal de Louvain)
-------------------------------------
1. Cada nodo empieza en su propia comunidad.
2. Se repite hasta que nada cambie:
   - Se recorren los nodos en orden aleatorio.
   - Para cada nodo, se prueba moverlo a la comunidad de cada vecino.
   - Se queda con el movimiento que más aumente Q.
3. Se devuelve la comunidad final de cada nodo.

Esta es una versión **sencilla y fácil de leer**: recalcula Q completo en cada
prueba. La versión de producción usa una fórmula incremental (ΔQ) y una fase
de agregación. Al final del archivo hay una lista de posibles mejoras.
"""

import random
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

# Tipos con nombres claros para que el código se lea mejor.
Adyacencia = Dict[str, List[Tuple[str, float]]]
Comunidades = Dict[str, int]


def simetrizar(adj: Adyacencia) -> Adyacencia:
    """Convierte un grafo dirigido en un grafo NO dirigido.

    Si existe la arista u -> v con peso w, se crea la arista no dirigida
    {u, v} con peso w. Si existen u -> v y v -> u, los pesos se suman.

    Para Louvain trabajamos con el grafo no dirigido porque una comunidad no
    tiene dirección: interesa si dos nodos están conectados, no en qué sentido.
    """
    # Paso 1: juntar los pesos de cada par no ordenado {a, b}.
    pesos_por_par: Dict[Tuple[str, str], float] = defaultdict(float)

    for origen, vecinos in adj.items():
        for destino, peso in vecinos:
            # sorted((origen, destino)) hace que (A, B) y (B, A) sean el
            # mismo par. Así no contamos dos veces la misma conexión.
            a, b = sorted((origen, destino))
            pesos_por_par[(a, b)] += peso

    # Paso 2: construir la lista de adyacencia no dirigida.
    adj_no_dirigida: Adyacencia = defaultdict(list)

    for (a, b), peso in pesos_por_par.items():
        # La arista existe en los dos sentidos.
        adj_no_dirigida[a].append((b, peso))
        adj_no_dirigida[b].append((a, peso))

    return dict(adj_no_dirigida)


def calcular_grados(adj_no_dirigida: Adyacencia) -> Dict[str, float]:
    """Calcula el grado ponderado de cada nodo.

    El grado de un nodo es la suma de los pesos de todas sus aristas.
    En un grafo no dirigido, cada arista aporta su peso a los dos extremos.
    """
    grados: Dict[str, float] = {}

    for nodo, vecinos in adj_no_dirigida.items():
        suma_de_pesos = 0.0
        for _, peso in vecinos:
            suma_de_pesos += peso
        grados[nodo] = suma_de_pesos

    return grados


def modularidad(
    adj_no_dirigida: Adyacencia,
    comunidades: Comunidades,
) -> float:
    """Calcula la modularidad Q de una partición.

    Se usa la fórmula por comunidad:

        Q = suma_c [ L_c / m - (K_c / (2m))^2 ]

    Parámetros:
        adj_no_dirigida: lista de adyacencia del grafo no dirigido.
        comunidades: diccionario {nodo: id_de_comunidad}.

    Retorna:
        Q, un número entre -0.5 y 1.0.
    """
    # Paso 1: grado de cada nodo.
    grados = calcular_grados(adj_no_dirigida)

    # Paso 2: m = peso total del grafo.
    # La suma de todos los grados cuenta cada arista dos veces, por eso / 2.
    suma_de_grados = 0.0
    for grado in grados.values():
        suma_de_grados += grado
    peso_total = suma_de_grados / 2.0

    if peso_total == 0.0:
        return 0.0

    # Paso 3: K_c = suma de grados de los nodos de cada comunidad.
    grado_por_comunidad: Dict[int, float] = defaultdict(float)
    for nodo, grado in grados.items():
        comunidad = comunidades[nodo]
        grado_por_comunidad[comunidad] += grado

    # Paso 4: L_c = peso de las aristas internas de cada comunidad.
    # Cada arista no dirigida se cuenta una sola vez. Para lograrlo, solo se
    # suma cuando nodo < vecino (o una vez si es un auto-loop).
    interno_por_comunidad: Dict[int, float] = defaultdict(float)

    for nodo, vecinos in adj_no_dirigida.items():
        for vecino, peso in vecinos:
            if comunidades[nodo] != comunidades[vecino]:
                continue  # la arista sale de la comunidad: no es interna

            if nodo < vecino:
                interno_por_comunidad[comunidades[nodo]] += peso
            elif nodo == vecino:
                # Un auto-loop suma dos veces al grado; para la parte interna
                # se cuenta una sola vez.
                interno_por_comunidad[comunidades[nodo]] += peso / 2.0

    # Paso 5: aplicar la fórmula para cada comunidad.
    q = 0.0
    for comunidad, grado_total in grado_por_comunidad.items():
        interno = interno_por_comunidad.get(comunidad, 0.0)
        parte_interna = interno / peso_total
        parte_esperada = (grado_total / (2.0 * peso_total)) ** 2
        q += parte_interna - parte_esperada

    return q


def louvain(
    adj_no_dirigida: Adyacencia,
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Comunidades:
    """Detecta comunidades maximizando la modularidad.

    Es la fase principal de Louvain:

    - Cada nodo empieza solo en su comunidad.
    - En cada pasada, se prueba mover cada nodo a la comunidad de sus vecinos.
    - Se acepta el movimiento solo si Q aumenta.
    - Se termina cuando una pasada completa no cambia nada.

    Parámetros:
        adj_no_dirigida: grafo no dirigido.
        semilla: semilla para que el resultado sea reproducible.
        max_pasadas: máximo de pasadas completas.

    Retorna:
        {nodo: id_de_comunidad}.
    """
    # random.Random(semilla) crea un generador propio. Así no cambiamos el
    # generador global de Python y el resultado es reproducible.
    generador = random.Random(semilla)

    nodos = list(adj_no_dirigida.keys())

    # Al inicio, cada nodo está en su propia comunidad.
    comunidades: Comunidades = {}
    for identificador, nodo in enumerate(nodos):
        comunidades[nodo] = identificador

    for numero_pasada in range(max_pasadas):
        hubo_cambio = False

        # Orden aleatorio para no favorecer siempre a los mismos nodos.
        generador.shuffle(nodos)

        for nodo in nodos:
            comunidad_actual = comunidades[nodo]
            mejor_comunidad = comunidad_actual
            mejor_q = modularidad(adj_no_dirigida, comunidades)

            # Comunidades de los vecinos. Usamos un set para no probar dos
            # veces la misma comunidad.
            comunidades_vecinas: Set[int] = set()
            for vecino, _ in adj_no_dirigida.get(nodo, []):
                comunidades_vecinas.add(comunidades[vecino])

            # Probamos cada comunidad vecina (en orden fijo para que sea
            # reproducible).
            for comunidad_vecina in sorted(comunidades_vecinas):
                if comunidad_vecina == comunidad_actual:
                    continue

                # Mover temporalmente el nodo.
                comunidades[nodo] = comunidad_vecina
                q_con_movimiento = modularidad(adj_no_dirigida, comunidades)

                if q_con_movimiento > mejor_q:
                    mejor_q = q_con_movimiento
                    mejor_comunidad = comunidad_vecina

                # Deshacer el movimiento antes de probar el siguiente.
                comunidades[nodo] = comunidad_actual

            # Si encontramos una comunidad mejor, dejamos el nodo allí.
            if mejor_comunidad != comunidad_actual:
                comunidades[nodo] = mejor_comunidad
                hubo_cambio = True

        # Si nadie se movió, ya no tiene sentido seguir.
        if not hubo_cambio:
            break

    return comunidades


def resumen_comunidades(
    comunidades: Comunidades,
    etiquetas: Optional[Dict[str, Dict[str, str]]] = None,
) -> Dict[int, List[str]]:
    """Agrupa los nodos por comunidad y devuelve nombres legibles.

    Parámetros:
        comunidades: {nodo: id_de_comunidad}.
        etiquetas: opcional, {dirección: {"nombre": ...}}.

    Retorna:
        {id_de_comunidad: [nombres de sus nodos]}.
    """
    grupos: Dict[int, List[str]] = defaultdict(list)

    for nodo, comunidad in comunidades.items():
        nombre = nodo
        if etiquetas is not None and nodo in etiquetas:
            nombre = etiquetas[nodo].get("nombre", nodo)
        grupos[comunidad].append(nombre)

    return dict(grupos)


def renumerar_comunidades(comunidades: Comunidades) -> Comunidades:
    """Cambia los IDs de comunidad por 0, 1, 2, ... en orden de aparición.

    Louvain devuelve IDs que son el índice original de un nodo (por ejemplo,
    300 o 692). Eso es correcto, pero para leer los resultados es más cómodo
    que las comunidades se llamen 0, 1, 2, ...

    No cambia la partición: solo cambia el nombre de cada grupo.
    """
    mapa_de_ids: Dict[int, int] = {}
    comunidades_nuevas: Comunidades = {}

    for nodo, id_viejo in comunidades.items():
        if id_viejo not in mapa_de_ids:
            mapa_de_ids[id_viejo] = len(mapa_de_ids)
        comunidades_nuevas[nodo] = mapa_de_ids[id_viejo]

    return comunidades_nuevas


def desde_aristas(aristas: List[Tuple[str, str, float]]) -> Adyacencia:
    """Construye el grafo no dirigido a partir de la lista compartida de aristas.

    La lista `grafo.aristas` viene de la carga única del dataset. Cada módulo
    recibe esa lista y construye su propia estructura; así no depende de la
    estructura de otro módulo.

    Si hay transferencias en ambos sentidos, sus pesos se suman.
    """
    # Primero construimos el grafo dirigido tal como viene en las aristas.
    adj_dirigida: Adyacencia = {}

    for origen, destino, peso in aristas:
        if origen not in adj_dirigida:
            adj_dirigida[origen] = []
        adj_dirigida[origen].append((destino, float(peso)))

    # Después lo convertimos a no dirigido.
    return simetrizar(adj_dirigida)


def louvain_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Comunidades:
    """Atajo: construye el grafo no dirigido y ejecuta Louvain.

    Es la función que usa el notebook. Recibe directamente `grafo.aristas`.
    """
    adj_no_dirigida = desde_aristas(aristas)
    return louvain(
        adj_no_dirigida,
        semilla=semilla,
        max_pasadas=max_pasadas,
    )


# Posibles mejoras (para cuando el proyecto crezca)
# 1. Calcular el cambio de modularidad ΔQ con una fórmula incremental, en
#    vez de recalcular Q completo en cada prueba. Eso hace el algoritmo mucho
#    más rápido en grafos grandes.
# 2. Agregar la segunda fase de Louvain: convertir cada comunidad en un
#    super-nodo y repetir el proceso. Es la versión completa del paper.
# 3. Agregar el parámetro de resolución γ, que permite encontrar comunidades
#    más grandes o más pequeñas.
# 4. Probar varias semillas y quedarse con la partición de mayor Q.
# 5. Comparar con Label Propagation y con `python-louvain` usando la misma
#    muestra y las mismas etiquetas de exchanges.
# 6. Para grafos de millones de aristas, usar arreglos/CSR en vez de
#    diccionarios de Python.


# Paso 1: construir el grafo no dirigido.
adj_no_dirigido = desde_aristas(grafo.aristas)

cantidad_nodos = len(adj_no_dirigido)
cantidad_aristas = 0
for vecinos in adj_no_dirigido.values():
    cantidad_aristas += len(vecinos)
cantidad_aristas = cantidad_aristas // 2

print("Grafo no dirigido:")
print(f"  nodos   : {cantidad_nodos:,}")
print(f"  aristas : {cantidad_aristas:,}")

# Paso 2: ejecutar Louvain con una semilla fija.
comunidades = louvain(adj_no_dirigido, semilla=42)

# Renumerar 0, 1, 2, ... para que sea más fácil leer los resultados.
comunidades = renumerar_comunidades(comunidades)

# Paso 3: medir la calidad de la partición.
Q = modularidad(adj_no_dirigido, comunidades)

print()
print(f"Modularidad Q = {Q:.3f}")
print(f"Número de comunidades = {len(set(comunidades.values())):,}")

# Paso 4: mostrar las comunidades más grandes.
grupos = resumen_comunidades(comunidades, grafo.etiquetas)

print()
print("Comunidades más grandes:")
for posicion, (id_comunidad, nombres) in enumerate(
    sorted(grupos.items(), key=lambda par: -len(par[1]))[:10],
    start=1,
):
    nombres_legibles = sorted(nombres, key=lambda n: (n.startswith("0x"), n))
    primeros = ", ".join(nombres_legibles[:5])
    if len(nombres) > 5:
        primeros += ", ..."
    print(f"  Comunidad {posicion:>2}: {len(nombres):>4} nodos | {primeros}")

# Paso 5: ver en qué comunidad cae cada exchange etiquetado.
print()
print("Comunidad de cada exchange etiquetado:")
for direccion, info in sorted(
    grafo.etiquetas.items(), key=lambda par: par[1]["nombre"]
):
    if direccion not in comunidades:
        continue
    id_comunidad = comunidades[direccion]
    tamano = len(grupos[id_comunidad])
    print(f"  {info['nombre']:<18} comunidad {id_comunidad:>3} ({tamano} nodos)")

### Experimento: ¿qué pasa si cambiamos la semilla?

Louvain es un algoritmo **heurístico**: el orden en que prueba los nodos
puede cambiar el resultado. Por eso se repite con varias semillas y se mira
si Q se mantiene estable.

In [ ]:
import matplotlib.pyplot as plt

semillas = [1, 7, 42, 123, 2024]
resultados = []

for semilla in semillas:
    comunidades_semilla = louvain(adj_no_dirigido, semilla=semilla)
    q = modularidad(adj_no_dirigido, comunidades_semilla)
    cuantas = len(set(comunidades_semilla.values()))
    resultados.append((semilla, q, cuantas))

print(f"{'Semilla':>8} {'Q':>8} {'Comunidades':>12}")
print("-" * 32)
for semilla, q, cuantas in resultados:
    print(f"{semilla:>8} {q:>8.3f} {cuantas:>12}")

mejor = max(resultados, key=lambda fila: fila[1])
print()
print(f"Mejor semilla: {mejor[0]} con Q = {mejor[1]:.3f}")

plt.figure(figsize=(6, 4))
plt.bar(
    [str(fila[0]) for fila in resultados],
    [fila[1] for fila in resultados],
    color="seagreen",
)
plt.xlabel("Semilla")
plt.ylabel("Modularidad Q")
plt.title("Sensibilidad de Louvain a la semilla")
plt.ylim(0, 1)
plt.show()

### Validación complementaria de Louvain (aporte de Pablo)

Esta sección agrega la versión completa de Louvain que aportó Pablo:

- fase local + **agregación** (super-nodos),
- varios **reinicios** y se queda con la mejor Q,
- **NMI** para comparar particiones,
- barrido de **resolución**.

Es una validación opcional. La explicación principal de Louvain sigue siendo
la de la sección anterior.

In [ ]:
"""
comunidades_liquidez.py
=======================
Tarea 3 — Comunidades de liquidez mediante Louvain.

Responsable: Juan Pablo Álvarez.

Es el código de la Tarea 3 del notebook entregable
(`proyecto_grafos_ethereum.ipynb`), sacado a un módulo para poder probarlo
con pytest. Solo usa la librería estándar.

Grafo: no dirigido y ponderado. Nodos = exchanges + contrapartes principales;
peso de {u, v} = número de transferencias stablecoin entre u y v (en ambos
sentidos). Representación: diccionario de diccionarios {u: {v: peso}}.

Funciones del notebook (sin cambios de lógica):
    modularity, louvain_once, louvain, label_propagation

Funciones nuevas para validar y analizar sensibilidad:
    grafo_no_dirigido      construye {u: {v: peso}} desde (src, dst, peso)
    modularidad_definicion Q con la doble suma de la definición, O(n^2)
    nmi                    información mutua normalizada entre particiones
    estabilidad_reinicios  Q y NMI de cada reinicio de Louvain
    barrido_resolucion     comunidades y Q para varios valores de gamma
"""

import math
import random
from collections import Counter, defaultdict
from typing import Dict, Hashable, Iterable, List, Tuple

def modularity_completa(adj, loops, comm, resolution=1.0):
    m = sum(sum(adj[u].values()) for u in adj) / 2 + sum(loops.values())
    if m == 0:
        return 0.0
    in_c = defaultdict(float)
    tot_c = defaultdict(float)
    for u in adj:
        cu = comm[u]
        tot_c[cu] += sum(adj[u].values()) + 2 * loops.get(u, 0.0)
        in_c[cu] += loops.get(u, 0.0)
        for v, w in adj[u].items():
            if v > u and comm[v] == cu:
                in_c[cu] += w
    return sum(in_c[c] / m - resolution * (tot_c[c] / (2 * m)) ** 2
               for c in set(comm.values()))


def louvain_once(orig_nodes, adj0, resolution=1.0, seed=0, max_levels=20):
    rng = random.Random(seed)
    adj = {u: dict(adj0[u]) for u in orig_nodes}
    loops = {u: 0.0 for u in orig_nodes}
    membership = {u: u for u in orig_nodes}
    levels = []

    for level in range(max_levels):
        m = sum(sum(adj[u].values()) for u in adj) / 2 + sum(loops.values())
        if m == 0:
            break
        k = {u: sum(adj[u].values()) + 2 * loops[u] for u in adj}
        tot = dict(k)
        comm = {u: u for u in adj}
        order = sorted(adj)
        moves_total = 0

        for _ in range(200):
            rng.shuffle(order)
            moved = 0
            for u in order:
                ci = comm[u]
                nc = defaultdict(float)
                for v, w in adj[u].items():
                    if v != u:
                        nc[comm[v]] += w
                tot[ci] -= k[u]
                best_c, best_gain = ci, 0.0
                for c, w in nc.items():
                    gain = w / m - resolution * tot.get(c, 0.0) * k[u] / (2 * m * m)
                    if gain > best_gain + 1e-12:
                        best_gain, best_c = gain, c
                if best_c != ci:
                    comm[u] = best_c
                    tot[best_c] += k[u]
                    moved += 1
                else:
                    tot[ci] += k[u]
            moves_total += moved
            if moved == 0:
                break

        membership = {o: comm[v] for o, v in membership.items()}
        levels.append({"level": level, "n_comms": len(set(comm.values())),
                       "modularity": modularity_completa(adj, loops, comm, resolution),
                       "moves": moves_total})
        if moves_total == 0:
            break

        new_adj = defaultdict(dict)
        new_loops = defaultdict(float)
        for u in adj:
            cu = comm[u]
            new_adj.setdefault(cu, {})
            new_loops[cu] += loops[u]
            for v, w in adj[u].items():
                if v < u:
                    continue
                cv = comm[v]
                if cu == cv:
                    new_loops[cu] += w
                else:
                    new_adj[cu][cv] = new_adj[cu].get(cv, 0.0) + w
                    new_adj[cv][cu] = new_adj[cv].get(cu, 0.0) + w
        adj = {u: dict(new_adj[u]) for u in new_adj}
        loops = {u: new_loops.get(u, 0.0) for u in adj}
        if len(adj) >= len(orig_nodes):
            break

    adj_orig = {u: dict(adj0[u]) for u in orig_nodes}
    loops0 = {u: 0.0 for u in orig_nodes}
    return membership, modularity_completa(adj_orig, loops0, membership, resolution), levels


def louvain_completo(nodes, adj, resolution=1.0, n_restarts=10, base_seed=0):
    best = None
    for r in range(n_restarts):
        mem, q, levels = louvain_once(nodes, adj, resolution, base_seed + r)
        if best is None or q > best[1] + 1e-12:
            best = (mem, q, levels, r)
    return best


def label_propagation(nodes, adj, max_iter=100, seed=42):
    rng = random.Random(seed)
    labels = {u: u for u in nodes}
    order = sorted(nodes)
    for _ in range(max_iter):
        rng.shuffle(order)
        changed = 0
        for u in order:
            counts = defaultdict(float)
            for v, w in adj[u].items():
                counts[labels[v]] += w
            if not counts:
                continue
            best = max(counts.values())
            candidates = [c for c, w in counts.items() if abs(w - best) < 1e-12]
            new = labels[u] if labels[u] in candidates else min(candidates)
            if new != labels[u]:
                labels[u] = new
                changed += 1
        if changed == 0:
            break
    return labels



# Construcción del grafo

def grafo_no_dirigido(aristas: Iterable[Tuple[str, str, float]]) -> Dict[str, Dict[str, float]]:
    """Grafo no dirigido {u: {v: peso}} sumando ambos sentidos.

    Igual que la celda de construcción del notebook: se ignoran los autolazos
    y los pesos de u->v y v->u se suman en la arista {u, v}.
    """
    g: Dict[str, Dict[str, float]] = defaultdict(dict)
    for s, d, w in aristas:
        if s == d:
            continue
        w = float(w)
        g[s][d] = g[s].get(d, 0.0) + w
        g[d][s] = g[d].get(s, 0.0) + w
    return {u: dict(vecinos) for u, vecinos in g.items()}



# Validación

def modularidad_definicion(adj: Dict[str, Dict[str, float]], comm: Dict[str, Hashable]) -> float:
    """Modularidad con la fórmula de la definición (doble suma sobre i, j).

        Q = 1/(2m) * sum_ij [A_ij - k_i k_j / (2m)] * delta(c_i, c_j)

    Es O(n^2): solo sirve para comprobar `modularity` en grafos pequeños.
    """
    nodos = list(adj)
    k = {u: sum(adj[u].values()) for u in nodos}
    dos_m = sum(k.values())
    if dos_m == 0:
        return 0.0
    q = 0.0
    for i in nodos:
        for j in nodos:
            if comm[i] == comm[j]:
                q += adj[i].get(j, 0.0) - k[i] * k[j] / dos_m
    return q / dos_m


def nmi(particion_a: Dict[str, Hashable], particion_b: Dict[str, Hashable]) -> float:
    """Información mutua normalizada entre dos particiones de los mismos nodos.

    NMI = 2 I(A; B) / (H(A) + H(B)), en [0, 1]: 1 = particiones idénticas
    (salvo el nombre de las comunidades), 0 = independientes.
    """
    nodos = list(particion_a)
    n = len(nodos)
    if n == 0:
        return 1.0
    ca = Counter(particion_a[u] for u in nodos)
    cb = Counter(particion_b[u] for u in nodos)
    cab = Counter((particion_a[u], particion_b[u]) for u in nodos)
    h_a = -sum(c / n * math.log(c / n) for c in ca.values())
    h_b = -sum(c / n * math.log(c / n) for c in cb.values())
    if h_a + h_b == 0:
        return 1.0
    info = sum(c / n * math.log(c * n / (ca[a] * cb[b])) for (a, b), c in cab.items())
    return 2 * info / (h_a + h_b)



# Estabilidad y sensibilidad

def estabilidad_reinicios(nodes, adj, n_restarts=10, base_seed=0, resolution=1.0) -> List[Dict[str, float]]:
    """Ejecuta Louvain con varias semillas y compara cada resultado con el mejor.

    Retorna una fila por reinicio: semilla, comunidades, Q y NMI contra la
    partición de mayor Q.
    """
    corridas = []
    for r in range(n_restarts):
        mem, q, _ = louvain_once(nodes, adj, resolution, base_seed + r)
        corridas.append((base_seed + r, mem, q))
    mejor = max(corridas, key=lambda t: t[2])[1]
    return [
        {"semilla": s, "comunidades": len(set(m.values())), "Q": q, "NMI_vs_mejor": nmi(m, mejor)}
        for s, m, q in corridas
    ]


def barrido_resolucion(nodes, adj, gammas=(0.5, 1.0, 1.5, 2.0, 3.0), n_restarts=3) -> List[Dict[str, float]]:
    """Louvain para varios valores de resolución gamma.

    gamma < 1 favorece comunidades grandes; gamma > 1, comunidades pequeñas.
    Se reporta la Q con gamma = 1 para poder comparar las particiones.
    """
    loops = {u: 0.0 for u in nodes}
    filas = []
    for g in gammas:
        mem, _, _, _ = louvain_completo(nodes, adj, resolution=g, n_restarts=n_restarts)
        filas.append({
            "gamma": g,
            "comunidades": len(set(mem.values())),
            "Q_gamma_1": modularity_completa(adj, loops, mem, 1.0),
        })
    return filas


# Louvain completo: fase local + agregación + reinicios + resolución.
adj_t3 = grafo_no_dirigido(grafo.aristas)
nodos_t3 = sorted(adj_t3)

mem_completo, q_completo, niveles, mejor_reinicio = louvain_completo(
    nodos_t3, adj_t3, n_restarts=5, base_seed=42
)

print("Louvain completo (con agregación):")
print("  comunidades:", len(set(mem_completo.values())))
print("  Q          :", round(q_completo, 3))
print("  niveles    :", len(niveles))
for nivel in niveles:
    print("   ", nivel)

print("Q de la versión simple:", round(modularidad(adj_no_dirigido, comunidades), 3))

print("Estabilidad entre reinicios:")
for fila in estabilidad_reinicios(nodos_t3, adj_t3, n_restarts=5):
    print(
        f"  semilla {fila['semilla']}: Q={fila['Q']:.3f}, "
        f"comunidades={fila['comunidades']}, NMI={fila['NMI_vs_mejor']:.3f}"
    )

## Tarea 4 — Kruskal: árbol de expansión mínima

**Responsabilidad:** grupal

**Estructura de datos:** lista de aristas + Union-Find.

Se ordenan las aristas por peso y se agregan solo si no forman un ciclo,
usando **Union-Find**. El resultado tiene `|V|-1` aristas.

In [ ]:
"""
arbol_expansion.py
==================
Tarea 4 — Árbol de expansión mínima (MST).

Problema: dado un grafo ponderado, encontrar el conjunto de aristas que
conecta TODOS los nodos con el MENOR costo total posible. Ese conjunto
siempre forma un árbol (no tiene ciclos) y tiene exactamente |V|-1 aristas.

Algoritmo elegido: Kruskal.

    1. Ordenar las aristas por peso de menor a mayor.
    2. Recorrerlas en orden y agregar cada arista SOLO si no forma un ciclo.
    3. Para detectar ciclos usamos Union-Find (conjuntos disjuntos).

Union-Find es una estructura de datos clásica que permite saber rápido si
dos nodos ya están conectados (están en el mismo conjunto) y unirlos.
"""

from typing import Dict, List, Set, Tuple


class UnionFind:
    """Conjuntos disjuntos con compresión de caminos y unión por rango."""

    def __init__(self, nodos: Set[str]) -> None:
        # Cada nodo apunta a su "padre". Al inicio, cada uno es su propio padre.
        self.padre: Dict[str, str] = {v: v for v in nodos}
        # El "rango" es una cota de la altura del árbol de cada conjunto.
        self.rango: Dict[str, int] = {v: 0 for v in nodos}

    def encontrar(self, x: str) -> str:
        """Devuelve el representante (raíz) del conjunto al que pertenece x."""
        # Compresión de caminos: hacemos que cada nodo apunte directo a la raíz,
        # así las siguientes búsquedas son más rápidas.
        if self.padre[x] != x:
            self.padre[x] = self.encontrar(self.padre[x])
        return self.padre[x]

    def unir(self, x: str, y: str) -> bool:
        """Une los conjuntos de x e y.

        Retorna:
            True si estaban en conjuntos distintos (y los unió),
            False si ya estaban conectados (agregar la arista crearía un ciclo).
        """
        rx = self.encontrar(x)
        ry = self.encontrar(y)
        if rx == ry:
            return False   # ya conectados: esta arista formaría un ciclo

        # Unión por rango: el árbol más "bajo" cuelga del más "alto".
        if self.rango[rx] < self.rango[ry]:
            rx, ry = ry, rx
        self.padre[ry] = rx
        if self.rango[rx] == self.rango[ry]:
            self.rango[rx] += 1
        return True


def aristas_no_dirigidas(aristas: List[Tuple[str, str, float]]) -> List[Tuple[str, str, float]]:
    """Convierte aristas dirigidas en no dirigidas.

    Si aparecen (a, b, w1) y (b, a, w2), las juntamos en una sola arista
    {a, b} con peso w1 + w2.
    """
    pesos: Dict[Tuple[str, str], float] = {}
    for a, b, w in aristas:
        clave = tuple(sorted((a, b)))
        pesos[clave] = pesos.get(clave, 0.0) + w
    return [(a, b, w) for (a, b), w in pesos.items()]


def kruskal(
    nodos: Set[str],
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Construye el árbol de expansión mínima con el algoritmo de Kruskal.

    Parámetros:
        nodos: conjunto de todos los vértices.
        aristas: lista de aristas NO dirigidas (a, b, peso).

    Retorna:
        (mst, costo_total) donde mst es la lista de aristas del árbol.
    """
    # 1. Ordenar las aristas por peso, de menor a mayor.
    ordenadas = sorted(aristas, key=lambda e: e[2])

    # 2. Estructura Union-Find para detectar ciclos.
    uf = UnionFind(nodos)

    mst: List[Tuple[str, str, float]] = []
    costo = 0.0
    objetivo = len(nodos) - 1   # un árbol con |V| nodos tiene |V|-1 aristas

    # 3. Recorrer las aristas en orden.
    for a, b, w in ordenadas:
        if len(mst) == objetivo:
            break   # ya tenemos el árbol completo

        if uf.unir(a, b):
            mst.append((a, b, w))
            costo += w

    return mst, costo


def resumen_mst(mst: List[Tuple[str, str, float]]) -> Dict[str, float]:
    """Resumen simple del MST: número de aristas y costo total."""
    return {
        "aristas": len(mst),
        "costo_total": sum(w for _, _, w in mst),
    }


def kruskal_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Kruskal a partir de la lista compartida de aristas.

    Convierte las aristas dirigidas en no dirigidas (sumando pesos si hay
    transferencias en ambos sentidos) y calcula el MST.
    """
    no_dirigidas = aristas_no_dirigidas(
        [(origen, destino, float(peso)) for origen, destino, peso in aristas]
    )
    nodos: Set[str] = set()
    for origen, destino, _ in no_dirigidas:
        nodos.add(origen)
        nodos.add(destino)
    return kruskal(nodos, no_dirigidas)


mst, costo = kruskal_desde_aristas(grafo.aristas)
print(f"MST: {len(mst)} aristas, costo total = {costo:,.2f}\n")
print("Primeras 20 aristas del MST:")
for a, b, w in sorted(mst, key=lambda e: e[2])[:20]:
    print(f"  {grafo.nombre(a):<18} -- {grafo.nombre(b):<18} {w:,.2f}")

## Estadísticas del dataset completo

Esta sección sí usa las **38.901.039 transferencias**. DuckDB lee el Parquet
por columnas y calcula los totales sin cargar todo en RAM.

In [ ]:
import os
import time

if not RUTA_PARQUET:
    print("Sección omitida: no encontré el Parquet completo.")
    print("Súbelo a Drive como MyDrive/ED2/transferencias.parquet.")
else:
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"], check=True)
        import duckdb

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")
    con.execute("SET preserve_insertion_order=false")

    t0 = time.perf_counter()
    r = con.execute(f"""
        SELECT count(*) filas,
               count(DISTINCT token_address) tokens,
               min(block_number) bmin,
               max(block_number) bmax
        FROM read_parquet('{RUTA_PARQUET}')
    """).fetchone()
    print(f"Transferencias : {r[0]:,}")
    print(f"Tokens         : {r[1]}")
    print(f"Bloques        : {r[2]:,} .. {r[3]:,}")
    print(f"Tiempo         : {time.perf_counter()-t0:.1f} s")

    print("\nTop 10 tokens por transferencias:")
    for tok, n in con.execute(f"""
        SELECT token_address, count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY 1 ORDER BY n DESC LIMIT 10
    """).fetchall():
        print(f"  {tok}  {n:>10,}")

    print("\nActividad mensual aproximada:")
    for mes, n in con.execute(f"""
        SELECT strftime(to_timestamp(epoch(TIMESTAMP '2018-10-08') + (block_number-6525201)*13.5), '%Y-%m') mes,
               count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY mes ORDER BY mes
    """).fetchall():
        print(f"  {mes}: {n:>10,}")

## Conclusiones

1. **Un mismo dataset admite varias preguntas.** Bellman-Ford responde
   caminos, PageRank responde importancia, Louvain responde grupos y Kruskal
   responde conectividad mínima.

2. **Las estructuras de datos importan.** La lista de adyacencia, Union-Find y
   los diccionarios hacen que los algoritmos sean simples y legibles.

3. **La escala importa.** El análisis completo usa 38,9 M de transferencias con
   DuckDB; los algoritmos en Python puro se ejecutan sobre una muestra
   conectada de USDC.

4. **La muestra no reemplaza al dataset.** El Parquet conserva las 38.901.039
   filas; la muestra es solo para que los algoritmos terminen en Colab.

5. **Todo es reproducible.** La muestra tiene semilla 42 y el pipeline está
   documentado en el repositorio.